# FS4A_SGN_stage_expression

Existing analysis and drawing code is collected in this one panel notebook. Original shared methods may produce the complete original group of figures. The final drawing section below retains this panel's existing layout.

Configure the existing ROOT placeholders for the inputs and outputs of each original section. The primary notebook cells retain their original order and numeric positions. Standalone programs and source helpers are complete independent source sections; their original command-line, file, namespace, and kernel assumptions remain in force. No new Run All workflow or runtime worker generator is added.

---

## SGN selected-gene age curves

Seven requested genes, using the exact VGN-lines aggregation and layout. The legend contains only CVG, VGN and SGN; colored individual-library points remain visible. The original B–D results are retained. Frozen source files are recorded in the new SGN-lines.h5; that file also supports independent replotting.

In [ ]:

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")

# Cleanup must never interpret the project/current directory as a temporary target.
def _release_cleanup_target(path):
    candidate = Path(path).resolve()
    current = Path.cwd().resolve()
    if candidate == current or candidate in current.parents or candidate == Path(candidate.anchor):
        raise RuntimeError("Configure the owned temporary cleanup path before running this section.")
    return path

#11. =========== SGN selected-gene age curves
# Append-only analysis; the prior B/C/D figures and two H5 files are protected.
from pathlib import Path
import os, sys, json, hashlib, platform, subprocess, shutil
# Set this storage path independently before running.
SGN_LINE_ROOT = ROOT
SGN_LINE_GENES = ('Foxo6','Zfpm2','Neurod2','Nhlh1','Neurod1','Prox1','Insm1')
# Set this storage path independently before running.
SGN_LINE_FIGURE_DIR = ROOT
# Set this storage path independently before running. SGN-lines.h5.
SGN_LINE_RESULT = ROOT
# Set this storage path independently before running. 05_fig4B-D_SGN.ipynb. Use the supplied fig4_SGN_program_analysis.ipynb source.
SGN_LINE_NOTEBOOK = ROOT
# Set this storage path independently before running.
SGN_LINE_WORK = ROOT
SGN_LINE_FIGSIZE, SGN_LINE_DPI = (15.2,6.4),600
REBUILD_SGN_LINES = False  # True after changing data preparation; appearance changes only redraw.
for p in (SGN_LINE_WORK,SGN_LINE_FIGURE_DIR,SGN_LINE_RESULT.parent): p.mkdir(parents=True,exist_ok=True)
# Set this storage path independently before running.
os.environ['MPLCONFIGDIR']=str(ROOT)
sys.dont_write_bytecode=True
import h5py
import numpy as np
import pandas as pd
import nbformat
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator
from IPython.display import display,Image
from urllib.parse import quote,unquote
SGN_LINE_DATA_SOURCE_SHA256 = 'b712bd1fc4c64b302907823fc0df88428256e84c447e4a1d2a56a18f8dd2f416'


SGN_LINE_INPUTS = {
    # Set this storage path independently before running. RNA-screen.h5.
    'RNA': 'ROOT',
    # Set this storage path independently before running. activity-screen.h5.
    'identity': 'ROOT',
    # Set this storage path independently before running. VGN520-common-counts.npz.
    'VGN_membership': 'ROOT',
}
SGN_EXPECTED_AGES = {
    'CVG': [9.5, 10.5, 11.5, 13.5],
    'VGN': [11.5, 13.5],
    'SGN': [11.5, 13.5, 14.5, 15.5, 16.5, 17.5, 18.5],
}
sys.dont_write_bytecode = True


def _sgn_line_digest(path):
    value = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()


def _sgn_line_array(node):
    if isinstance(node, h5py.Dataset):
        return node.asstr()[()] if node.dtype.kind in 'OUS' else node[()]
    categories = _sgn_line_array(node['categories'])
    codes = node['codes'][()]
    result = np.full(len(codes), '', dtype=object)
    valid = codes >= 0
    result[valid] = categories[codes[valid]]
    return result


def _sgn_line_frame(group):
    return pd.DataFrame({key: _sgn_line_array(group[key]) for key in group})


def _sgn_line_bool(values):
    values = np.asarray(values)
    return values if values.dtype.kind == 'b' else np.isin(values.astype(str), ['True', 'true', '1'])


#2. =========== Native library pseudobulk, then equal-library/equal-source means
def prepare_sgn_lines(root, genes=SGN_LINE_GENES):
    """Return native RNA tables; no disk writes, inference, normalization refit or plotting.

    The flat ``line_units``, ``line_source`` and ``line_points`` tables support
    the existing VGN ``render_lines(data)`` directly; each table is stored once.
    """
    root, genes = Path(root).resolve(), tuple(map(str, genes))
    assert genes and len(set(genes)) == len(genes), 'Gene names must be nonempty and unique'
    paths = {key: root / value for key, value in SGN_LINE_INPUTS.items()}
    hashes = {str(path.relative_to(root)): _sgn_line_digest(path) for path in paths.values()}
    with h5py.File(paths['RNA'], 'r') as handle:
        obs = _sgn_line_frame(handle['cells'])
        tfs = handle['TFs'].asstr()[()]
        positions = pd.Index(tfs).get_indexer(genes)
        if np.any(positions < 0):
            absent = np.asarray(genes)[positions < 0].tolist()
            raise ValueError(f'Requested genes absent from the frozen native-TF axis: {absent}')
        # HDF5 fancy indexing requires monotonically increasing source columns.
        order = np.argsort(positions)
        counts = np.full((len(obs), len(genes)), -1, dtype=np.int64)
        counts[:, order] = handle['counts'][:, positions[order]]
        native_measured = np.zeros(counts.shape, dtype=bool)
        native_measured[:, order] = handle['cell_TF_measured'][:, positions[order]]
        vgn = _sgn_line_frame(handle['VGN_native_library_pseudobulk'])
    with h5py.File(paths['identity'], 'r') as handle:
        aobs = _sgn_line_frame(handle['obs'])
        np.testing.assert_array_equal(obs.cell_id, handle['cells'].asstr()[()])
        np.testing.assert_array_equal(obs.cell_id, aobs['_index'])
    assert len(obs) == 7355 and obs.cell_id.is_unique
    np.testing.assert_array_equal(native_measured, counts >= 0)
    assert np.all(obs.library_total > 0)
    good = (_sgn_line_bool(obs.source_state_evaluation_allowed)
            & ~_sgn_line_bool(obs.source_rna_mixture_review_flag))
    obs['identity'] = np.where(obs.study_id.eq('GSE178931'),
        np.where(aobs.cell_type_plot.eq('Differentiating SGN'), 'SGN', 'CVG'),
        np.where(obs.study_id.eq('GSE231513'), 'CVG-like', 'SGN'))

    with np.load(paths['VGN_membership'], allow_pickle=False) as handle:
        reference = pd.DataFrame({'cell': handle['cells'], 'stage': handle['stage'],
            'library': handle['library_id'], 'total': handle['full_endogenous_total']})
    assert len(reference) == 520 and reference.cell.is_unique
    assert not set(reference.cell).intersection(obs.cell_id)
    assert (reference.total > 0).all()
    vgn_expected = reference.groupby(['stage', 'library']).agg(
        cells=('cell', 'size'), full_endogenous_total_sum=('total', 'sum'))

    records, missing_units = [], []
    for (study, stage, identity, library), part in obs.loc[good].groupby(
            ['study_id', 'stage_standardized', 'identity', 'library_id']):
        ii = part.index.to_numpy()
        total = float(part.library_total.sum())
        values = counts[ii]
        measured = (values >= 0).all(axis=0)
        np.testing.assert_array_equal(measured, (values >= 0).any(axis=0))
        for j, gene in enumerate(genes):
            if not measured[j]:
                missing_units.append({'gene': gene, 'study': study, 'stage': stage,
                    'identity': identity, 'library': library, 'cells': len(ii),
                    'reason': 'Native source feature was not measured; omitted, never zero-filled'})
                continue
            cpm = float(values[:, j].sum() / total * 1e6)
            records.append({'gene': gene, 'study': study, 'stage': stage,
                'identity': identity, 'library': library, 'cells': len(ii),
                'expr': float(np.log10(1 + cpm))})

    vgn_selected = vgn.loc[vgn.TF.isin(genes)].copy()
    assert not vgn_selected.duplicated(['TF', 'study', 'stage', 'library_id']).any()
    for gene in genes:
        assert len(vgn_selected.loc[vgn_selected.TF.eq(gene)]) == len(vgn_expected)
    for _, row in vgn_selected.iterrows():
        expected = vgn_expected.loc[(row.stage, row.library_id)]
        assert int(row.cells) == int(expected.cells)
        assert float(row.full_endogenous_total_sum) == float(expected.full_endogenous_total_sum)
        np.testing.assert_allclose(row.log10_1p_full_endogenous_pseudobulk_CPM,
            np.log10(1 + row.full_endogenous_pseudobulk_CPM), atol=1e-12, rtol=0)
        records.append({'gene': row.TF, 'study': row.study, 'stage': row.stage,
            'identity': 'VGN', 'library': row.library_id, 'cells': int(row.cells),
            'expr': float(row.log10_1p_full_endogenous_pseudobulk_CPM)})

    units = pd.DataFrame(records)
    assert not units.duplicated(['gene', 'study', 'stage', 'identity', 'library']).any()
    assert np.isfinite(units.expr).all()
    units['age'] = units.stage.str[1:].astype(float)
    units['display_type'] = units.identity.replace({'CVG-like': 'CVG'})
    # These two means deliberately follow _vgn_line_data, including log before averaging.
    source = units.groupby(['gene', 'study', 'stage', 'age', 'display_type'],
                           as_index=False).expr.mean()
    points = source.groupby(['gene', 'stage', 'age', 'display_type'],
                            as_index=False).expr.mean()
    for gene in genes:
        for identity, ages in SGN_EXPECTED_AGES.items():
            selected = points.loc[points.gene.eq(gene)
                & points.display_type.eq(identity)].sort_values('age')
            assert selected.age.tolist() == ages
    assert points.expr.between(0, 4.1).all()
    missing = {gene: int((counts[:, j] < 0).sum()) for j, gene in enumerate(genes)}
    missing_table = pd.DataFrame(missing_units, columns=[
        'gene', 'study', 'stage', 'identity', 'library', 'cells', 'reason'])
    for relative, expected in hashes.items():
        assert _sgn_line_digest(root / relative) == expected, relative
    if genes == SGN_LINE_GENES:
        assert (len(units), len(source), len(points)) == (338, 109, 91)
        assert missing == {g: (4616 if g == 'Zfpm2' else 0) for g in genes}
        assert len(missing_table) == 12 and missing_table.study.eq('GSE195500').all()

    lines = {
        'genes': np.asarray(genes), 'line_units': units, 'line_source': source,
        'line_points': points, 'line_missing_source_features': missing,
        'missing_library_genes': missing_table,
        'source_hashes': hashes,
        'methods': {
            # Set this storage path independently before running. 05_fig4B-D_VGN.ipynb. Use the supplied fig4_VGN_program_analysis.ipynb source.
            'adapted_from': 'Code_def/05_fig4B-D_VGN.ipynb::_vgn_line_data',
            'unit': 'log10(1+sum raw counts/sum full endogenous library totals*1e6)',
            'averaging': 'Equal library weights within source/stage/type, then equal source weights',
            'E10.5': 'CVG-like source identity retained; ordinary CVG display type and marker',
            'VGN': 'Independent curated VGN520; no candidate tree-arm substitution',
            'missing': 'Native source feature count=-1; absent library-gene rows omitted, never replaced by zero',
            'scope': 'Original frozen identities and quality flags; no new marker selection, clustering, fitting or expression imputation',
            'limits': ['Source averaging does not remove platform effects.',
                       'Age curves are cross-sectional descriptive summaries, not lineage tracking.',
                       'GSM/library identity is not assumed to be an independently documented animal replicate.',
                       'E9.5 is 12 frozen atlas cells from one library.'],
        },
        'diagnostics': {'native_atlas_cells': 7355, 'curated_VGN_cells': 520,
            'library_gene_rows': len(units), 'source_gene_rows': len(source),
            'curve_points': len(points), 'source_inputs_unchanged': True,
            'all_seven_genes_in_requested_order': genes == SGN_LINE_GENES},
    }
    return lines


# Serialization for the single SGN-lines result
def _sgn_line_plain(value):
    if isinstance(value, np.generic): return value.item()
    if isinstance(value, np.ndarray): return value.tolist()
    if isinstance(value, Path): return str(value)
    raise TypeError(type(value).__name__)


def write_sgn_line_value(parent, key, value):
    if isinstance(value, pd.DataFrame):
        group = parent.create_group(key)
        group.attrs['kind'], group.attrs['columns'] = 'dataframe', json.dumps(value.columns.tolist())
        write_sgn_line_value(group, '_index', value.index.to_numpy())
        for column in value: write_sgn_line_value(group, column, value[column].to_numpy())
    elif isinstance(value, dict):
        group = parent.create_group(key); group.attrs['kind'] = 'dict'
        for name, item in value.items(): write_sgn_line_value(group, quote(str(name), safe=''), item)
    elif isinstance(value, np.ndarray):
        node = (parent.create_dataset(key, data=value.astype(str).astype(object), dtype=h5py.string_dtype())
                if value.dtype.kind in 'OUS' else parent.create_dataset(
                    key, data=value, compression='gzip' if value.ndim else None))
        node.attrs['kind'] = 'ndarray'
    else:
        node = parent.create_dataset(key, data=json.dumps(value, default=_sgn_line_plain), dtype=h5py.string_dtype())
        node.attrs['kind'] = 'json'


def read_sgn_line_value(node):
    kind = node.attrs['kind']
    if kind == 'dataframe':
        columns = json.loads(node.attrs['columns'])
        out = pd.DataFrame({key: read_sgn_line_value(node[key]) for key in columns})
        out.index = read_sgn_line_value(node['_index']); return out
    if kind == 'dict': return {unquote(key): read_sgn_line_value(node[key]) for key in node}
    if kind == 'json': return json.loads(node.asstr()[()])
    return node.asstr()[()] if node.dtype.kind in 'OUS' else node[()]




"""Strict vector PDF export for the Fig4 B/C/D Matplotlib canvas.

No analysis is performed. Each native Matplotlib label becomes one live Arial
text object, including the TF italic run in the two supported mathtext titles.
All geometry is written directly, without XObjects, glyph outlines or images.
The caller declares at most four data axes; their clip rectangles are collected
into one sibling viewport group per axis. Other artists must fit their axes.
"""
from pathlib import Path
import re
import sys
import math
import numpy as np
from matplotlib.backend_bases import RendererBase
from matplotlib.path import Path as MplPath
try:
    from reportlab.pdfgen import canvas
except ImportError:
    # Set this storage path independently before running.
    sys.path.append(str(ROOT))
    from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
_SGN_LINE_FONTS = {'normal': 'FigArial', 'italic': 'FigArialItalic', 'bold': 'FigArialBold', 'bolditalic': 'FigArialBoldItalic'}

def _sgn_line_register_fonts():
    for key, file in [('normal', 'arial.ttf'), ('italic', 'ariali.ttf'), ('bold', 'arialbd.ttf'), ('bolditalic', 'arialbi.ttf')]:
        if _SGN_LINE_FONTS[key] not in pdfmetrics.getRegisteredFontNames():
            # Set this storage path independently before running.
            pdfmetrics.registerFont(TTFont(_SGN_LINE_FONTS[key], str(ROOT / file)))

def _sgn_line_font(prop):
    bold = prop.get_weight() in ('bold', 'heavy', 'black', 'semibold', 'demibold') or (isinstance(prop.get_weight(), (int, float)) and prop.get_weight() >= 600)
    italic = prop.get_style() in ('italic', 'oblique')
    return _SGN_LINE_FONTS['bolditalic' if bold and italic else 'bold' if bold else 'italic' if italic else 'normal']

def _sgn_line_runs(text, prop, ismath):
    if not ismath:
        if '$' in text:
            raise ValueError('Unexpected dollar in plain label: ' + text)
        return [(text, _sgn_line_font(prop))]
    found = re.fullmatch('\\$\\\\it\\{([^{}]+)\\}([^$]*)\\$(.*)', text)
    if found is None:
        raise ValueError('Unsupported math label: ' + text)
    tf, inside, after = found.groups()
    return [(tf, _SGN_LINE_FONTS['italic']), (inside + after, _SGN_LINE_FONTS['normal'])]

class _SGNLineRecorder(RendererBase):

    def __init__(self, fig, agg, viewports, halo_labels):
        super().__init__()
        self.dpi = fig.dpi
        self.unit = 72 / fig.dpi
        self.agg = agg
        self.width, self.height = fig.bbox.size
        self.ops = []
        self.viewports = viewports
        self.halo_labels = halo_labels

    def points_to_pixels(self, p):
        return p / self.unit

    def flipy(self):
        return False

    def get_canvas_width_height(self):
        return (self.width, self.height)

    def get_text_width_height_descent(self, s, prop, ismath):
        runs = _sgn_line_runs(s, prop, ismath)
        width = sum((pdfmetrics.stringWidth(t, f, prop.get_size_in_points()) for t, f in runs)) / self.unit
        _, h, d = self.agg.get_text_width_height_descent(re.sub('\\$|\\\\it\\{|\\}', '', s), prop, False)
        return (width, h, d)

    def _clip(self, gc):
        if gc.get_clip_path()[0] is not None:
            raise ValueError('Nonrectangular artist clip is unsupported')
        b = gc.get_clip_rectangle()
        if b is None:
            return None
        for i, box in enumerate(self.viewports):
            if np.allclose(b.bounds, box, rtol=0, atol=1e-05):
                return i
        return None

    def draw_path(self, gc, path, transform, rgbFace=None):
        if gc.get_hatch():
            raise ValueError('Hatching/Pattern is forbidden')
        p = transform.transform_path(path)
        self.ops.append(('path', self._clip(gc), p, gc.get_rgb(), rgbFace, gc.get_linewidth(), gc.get_alpha(), gc.get_forced_alpha(), gc.get_dashes(), gc.get_capstyle(), gc.get_joinstyle()))

    def draw_text(self, gc, x, y, s, prop, angle, ismath=False, mtext=None):
        self.ops.append(('text', None, float(x), float(y), str(s), prop.copy(), float(angle), ismath, gc.get_rgb(), gc.get_alpha(), id(mtext) in self.halo_labels))

    def draw_image(self, *args, **kwargs):
        raise ValueError('Raster image is forbidden')

    def start_rasterizing(self):
        pass

    def stop_rasterizing(self):
        pass

    def start_filter(self):
        raise ValueError('Raster filter is forbidden')

    def stop_filter(self, *args):
        raise ValueError('Raster filter is forbidden')

def _sgn_line_pdf_path(pdf, path, unit):
    out = pdf.beginPath()
    current = None
    start = None
    for vertices, code in path.iter_segments(remove_nans=True, simplify=False, curves=True):
        v = np.asarray(vertices).reshape(-1, 2) * unit
        if code == MplPath.MOVETO:
            current = v[0]
            start = current
            out.moveTo(*current)
        elif code == MplPath.LINETO:
            current = v[0]
            out.lineTo(*current)
        elif code == MplPath.CURVE3:
            control, end = v
            a = current + (control - current) * 2 / 3
            b = end + (control - end) * 2 / 3
            out.curveTo(*a, *b, *end)
            current = end
        elif code == MplPath.CURVE4:
            out.curveTo(*v.ravel())
            current = v[-1]
        elif code == MplPath.CLOSEPOLY:
            out.close()
            current = start
        elif code == MplPath.STOP:
            break
        else:
            raise ValueError('Unknown path opcode ' + str(code))
    return out

def sgn_line_editable_pdf(fig, destination, *, data_axes=(), live_halo_width=0.2):
    """Write a strict vector PDF. Return declared clip boxes in PDF points.

    The artist canvas remains unchanged after return. Rasterization flags and
    white text path effects are replaced by a narrow live text fill+stroke
    (render mode 2) in the same single label object; no outlined font paths are
    emitted. The width is narrower than Agg's halo to preserve the black stems.
    """
    _sgn_line_register_fonts()
    if len(data_axes) > 4:
        raise ValueError('At most four sibling data viewports')
    fig.canvas.draw()
    agg = fig.canvas.get_renderer()
    viewports = [tuple(ax.bbox.bounds) for ax in data_axes]
    effects = []
    for artist in fig.findobj():
        if hasattr(artist, 'get_path_effects'):
            old = artist.get_path_effects()
            if old:
                effects.append((artist, old))
                artist.set_path_effects([])
    rec = _SGNLineRecorder(fig, agg, viewports, {id(a) for a, e in effects})
    try:
        fig.draw(rec)
    finally:
        for a, e in effects:
            a.set_path_effects(e)
    dst = Path(destination)
    dst.parent.mkdir(parents=True, exist_ok=True)
    pdf = canvas.Canvas(str(dst), pagesize=tuple(fig.get_size_inches() * 72), pageCompression=1, invariant=1, initialFontName=_SGN_LINE_FONTS['normal'])
    unit = rec.unit

    def emit(op):
        if op[0] == 'text':
            _, _, x, y, s, prop, angle, ismath, color, alpha, halo = op
            if not s:
                return
            pdf.saveState()
            pdf.setFillColorRGB(*color[:3])
            pdf.setFillAlpha(alpha)
            text = pdf.beginText()
            rad = math.radians(angle)
            if halo and live_halo_width > 0:
                pdf.setStrokeColorRGB(1, 1, 1)
                pdf.setStrokeAlpha(alpha)
                pdf.setLineWidth(float(live_halo_width))
                text.setTextRenderMode(2)
            text.setTextTransform(math.cos(rad), math.sin(rad), -math.sin(rad), math.cos(rad), x * unit, y * unit)
            for value, font in _sgn_line_runs(s, prop, ismath):
                if value:
                    text.setFont(font, prop.get_size_in_points())
                    text.textOut(value)
            pdf.drawText(text)
            pdf.restoreState()
            return
        _, _, path, stroke, face, lw, alpha, forced, dashes, cap, join = op
        pdf.saveState()
        pdf.setLineWidth(lw)
        pdf.setLineCap({'butt': 0, 'round': 1, 'projecting': 2}[cap])
        pdf.setLineJoin({'miter': 0, 'round': 1, 'bevel': 2}[join])
        offset, pattern = dashes
        pdf.setDash([] if pattern is None else [float(v) for v in pattern], 0.0 if offset is None else float(offset))
        if stroke is not None:
            pdf.setStrokeColorRGB(*stroke[:3])
            pdf.setStrokeAlpha(alpha if forced else stroke[3] if len(stroke) > 3 else alpha)
        if face is not None:
            pdf.setFillColorRGB(*face[:3])
            pdf.setFillAlpha(alpha if forced else face[3] if len(face) > 3 else alpha)
        pdf.drawPath(_sgn_line_pdf_path(pdf, path, unit), stroke=int(stroke is not None and lw > 0), fill=int(face is not None), fillMode=1)
        pdf.restoreState()
    done = set()
    for op in rec.ops:
        key = op[1]
        if key is None:
            emit(op)
            continue
        if key in done:
            continue
        done.add(key)
        pdf.saveState()
        p = pdf.beginPath()
        p.rect(*np.asarray(viewports[key]) * unit)
        pdf.clipPath(p, stroke=0, fill=0, fillMode=1)
        for candidate in rec.ops:
            if candidate[1] == key:
                emit(candidate)
        pdf.restoreState()
    pdf.showPage()
    pdf.save()
    if done != set(range(len(data_axes))):
        raise AssertionError('Declared clip has no artist')
    return {'expected_clips': len(done), 'clip_boxes_pt': (np.asarray(viewports) * unit).tolist(), 'editable_label_objects': sum((op[0] == 'text' for op in rec.ops)), 'label_strings': [op[4] for op in rec.ops if op[0] == 'text'], 'vector_paths': sum((op[0] == 'path' for op in rec.ops)), 'text_effects': 'White live text fill+stroke, single object; stroke width ' + str(live_halo_width) + ' pt.', 'halo_labels': len(effects)}

from PIL import Image as PILImage


def save_sgn_line_figure(fig, axes):
    fig.canvas.draw();renderer=fig.canvas.get_renderer()
    clipped=[]
    for text in fig.findobj(matplotlib.text.Text):
        if text.get_visible() and text.get_text():
            box=text.get_window_extent(renderer)
            if not fig.bbox.contains(*box.p0) or not fig.bbox.contains(*box.p1):clipped.append(text.get_text())
    assert not clipped,clipped
    # No data clip is needed: both centers and marker radii fit the declared axes.
    for ax in axes:
        for artist in ax.collections:
            xy=ax.transData.transform(np.asarray(artist.get_offsets(),float))
            radius=np.sqrt(np.max(artist.get_sizes()))/2*fig.dpi/72
            assert (xy[:,0]-radius >= ax.bbox.x0).all() and (xy[:,0]+radius <= ax.bbox.x1).all()
            assert (xy[:,1]-radius >= ax.bbox.y0).all() and (xy[:,1]+radius <= ax.bbox.y1).all()
        for artist in ax.lines:
            xy=ax.transData.transform(np.column_stack((artist.get_xdata(),artist.get_ydata())))
            radius=artist.get_markersize()/2*fig.dpi/72
            assert (xy[:,0]-radius >= ax.bbox.x0).all() and (xy[:,0]+radius <= ax.bbox.x1).all()
            assert (xy[:,1]-radius >= ax.bbox.y0).all() and (xy[:,1]+radius <= ax.bbox.y1).all()
    # Set this storage path independently before running. SGN-lines.png.
    png=ROOT;pdf=png.with_suffix('.pdf')
    fig.savefig(png,dpi=SGN_LINE_DPI,facecolor='white')
    qa=sgn_line_editable_pdf(fig,pdf,data_axes=())
    with PILImage.open(png) as im:
        assert im.size == (9120,3840)
        assert all(abs(x-SGN_LINE_DPI)<.1 for x in im.info['dpi'])
        qa.update(pixels=list(im.size),dpi=list(im.info['dpi']))
    qa.update(clipped_labels=clipped,legend_labels=['CVG','VGN','SGN'],
        colored_library_points_retained=True,figsize_in=list(SGN_LINE_FIGSIZE),
        library_alpha=.25,library_area=10.,
        palette={'CVG':'#8D77A1','VGN':'#5B85A7','SGN':'#69A18B'})
    plt.close(fig)
    return qa


def render_sgn_lines(data):
    FIGSIZE = SGN_LINE_FIGSIZE
    X_LIMITS, Y_LIMITS = (9.1, 18.9), (-.10, 4.10)
    X_TICKS, Y_TICKS = (9.5, 11.5, 13.5, 16.5, 18.5), (0, 1, 2, 3, 4)
    MEAN_LINE_WIDTH, MEAN_MARKER_SIZE = 1.65, 3.8
    LIBRARY_AREA, LIBRARY_ALPHA = 10.0, .25
    ORDER = ('CVG', 'VGN', 'SGN')
    COLORS = {'CVG': '#8D77A1', 'VGN': '#5B85A7', 'SGN': '#69A18B'}
    units, points = data['line_units'], data['line_points']
    plt.rcParams.update({'font.family': 'Arial', 'font.size': 9,
        'text.color': '#30343A', 'axes.labelcolor': '#30343A',
        'xtick.color': '#50555B', 'ytick.color': '#50555B', 'savefig.facecolor': 'white'})
    fig = plt.figure(figsize=FIGSIZE)
    grid = fig.add_gridspec(2, 4, left=.058, right=.985, bottom=.13, top=.94,
                           hspace=.53, wspace=.25)
    axes = []
    for j, gene in enumerate(SGN_LINE_GENES):
        row, col = (0, j) if j < 3 else (1, j - 3)
        ax = fig.add_subplot(grid[row, col])
        axes.append(ax)
        for identity in ORDER:
            u = units.loc[units.gene.eq(gene) & units.display_type.eq(identity)]
            p = points.loc[points.gene.eq(gene) & points.display_type.eq(identity)].sort_values('age')
            ax.scatter(u.age, u.expr, s=LIBRARY_AREA, color=COLORS[identity],
                       alpha=LIBRARY_ALPHA, linewidths=0, zorder=2)
            line, = ax.plot(p.age, p.expr, color=COLORS[identity], lw=MEAN_LINE_WIDTH,
                           marker='o', ms=MEAN_MARKER_SIZE, markeredgewidth=0, zorder=3)
            np.testing.assert_array_equal(line.get_ydata(), p.expr)
        ax.set_xlim(X_LIMITS); ax.set_ylim(Y_LIMITS)
        ax.set_xticks(X_TICKS, [f'E{x:g}' for x in X_TICKS])
        ax.xaxis.set_minor_locator(FixedLocator(np.arange(9.5, 19)))
        ax.set_yticks(Y_TICKS)
        if col: ax.tick_params(labelleft=False)
        ax.tick_params(which='major', length=2.6, width=.6, labelsize=8.5, pad=3)
        ax.tick_params(axis='x', which='minor', length=1.5, width=.45, color='#A5AAB0')
        ax.grid(axis='y', color='#EBECEE', linewidth=.55, zorder=0)
        for side in ('top', 'right'): ax.spines[side].set_visible(False)
        for side in ('left', 'bottom'):
            ax.spines[side].set_color('#A3A7AD'); ax.spines[side].set_linewidth(.65)
        ax.set_title(gene, fontstyle='italic', fontsize=12, pad=8)
    legend_ax = fig.add_subplot(grid[0, 3]); legend_ax.axis('off')
    handles = [Line2D([0], [0], color=COLORS[t], lw=MEAN_LINE_WIDTH, marker='o',
                      ms=4, markeredgewidth=0, label=t) for t in ORDER]
    line_legend = legend_ax.legend(handles=handles, loc='center', frameon=False, fontsize=9.5,
                     handlelength=2.2, labelspacing=1.0)
    fig.supylabel('RNA: mean log10(1 + pseudobulk CPM)', x=.012, y=.55, fontsize=10)
    fig.supxlabel('Embryonic age', x=.53, y=.035, fontsize=10)
    fig.canvas.draw(); renderer = fig.canvas.get_renderer()
    for ax in axes:
        boxes = [t.get_window_extent(renderer) for t in ax.get_xticklabels() if t.get_visible()]
        assert not any(a.overlaps(b) for i, a in enumerate(boxes) for b in boxes[i + 1:])
    assert [t.get_text() for t in line_legend.get_texts()] == list(ORDER)
    return save_sgn_line_figure(fig,axes)


# Frozen inputs and prior outputs are never overwritten by this appended block.
sgn_line_protected=[SGN_LINE_RESULT.parent/name for name in ('Activity.h5','Fig4B-D.h5')]
sgn_line_protected += [SGN_LINE_FIGURE_DIR/f'{tf}.{ext}' for tf in ('Foxo6','Zfpm2','Neurod2','Nhlh1','Neurod1') for ext in ('png','pdf')]
# Set this storage path independently before running.
sgn_line_protected += [ROOT/f'VGN-lines.{ext}' for ext in ('png','pdf')]
sgn_line_before={str(p.relative_to(SGN_LINE_ROOT)):_sgn_line_digest(p) for p in sgn_line_protected}
sgn_line_data=None
if SGN_LINE_RESULT.exists() and not REBUILD_SGN_LINES:
    with h5py.File(SGN_LINE_RESULT,'r') as h:
        if h.attrs.get('schema')=='SGN-seven-gene-lines-v1' and h['provenance/data_source_sha256'].asstr()[()]==SGN_LINE_DATA_SOURCE_SHA256:
            candidate=read_sgn_line_value(h['data'])
            if tuple(candidate['genes'])==SGN_LINE_GENES:
                present_hashes={p:_sgn_line_digest(SGN_LINE_ROOT/p) for p in candidate['source_hashes'] if (SGN_LINE_ROOT/p).exists()}
                if all(candidate['source_hashes'][p]==v for p,v in present_hashes.items()):sgn_line_data=candidate
if sgn_line_data is None:sgn_line_data=prepare_sgn_lines(SGN_LINE_ROOT,SGN_LINE_GENES)
np.testing.assert_array_equal(sgn_line_data['genes'],SGN_LINE_GENES)
with plt.rc_context():
    sgn_line_qa=render_sgn_lines(sgn_line_data)
# Set this storage path independently before running. validate_figure_output.py. Use the supplied fig4_validate_figure_output.py.
sgn_line_validator=ROOT
# Set this storage path independently before running.
sgn_line_dependencies=ROOT
# Set this storage path independently before running. python.exe.
sgn_line_check=subprocess.run([str(ROOT),str(sgn_line_validator),
    # Set this storage path independently before running. SGN-lines.png.
    str(ROOT),str(ROOT),
    '--expected-size-in','15.2x6.4','--expected-clips','0',
    # Set this storage path independently before running. pdftoppm.exe.
    '--pdftoppm',str(ROOT),
    # Set this storage path independently before running.
    '--render-dir',str(ROOT),'--render-dpi','150'],capture_output=True,text=True,errors='replace')
if sgn_line_check.returncode:raise RuntimeError(sgn_line_check.stdout+sgn_line_check.stderr)
sgn_line_qa.update(status='PASS',validator_stdout=sgn_line_check.stdout)
for p,old_hash in sgn_line_before.items():assert _sgn_line_digest(SGN_LINE_ROOT/p)==old_hash,p
sgn_line_notebook=nbformat.read(SGN_LINE_NOTEBOOK,as_version=4)
sgn_line_source=next(c.source for c in sgn_line_notebook.cells if c.cell_type=='code' and c.source.startswith('#11. =========== SGN selected-gene age curves'))
sgn_line_code_sha=hashlib.sha256(sgn_line_source.encode()).hexdigest()
# Set this storage path independently before running. SGN-lines.pending.h5.
sgn_line_pending=ROOT
with h5py.File(sgn_line_pending,'w') as h:
    h.attrs['schema']='SGN-seven-gene-lines-v1'
    write_sgn_line_value(h,'data',sgn_line_data)
    write_sgn_line_value(h,'figure',sgn_line_qa)
    p=h.create_group('provenance')
    p.create_dataset('data_source_sha256',data=SGN_LINE_DATA_SOURCE_SHA256,dtype=h5py.string_dtype())
    p.create_dataset('append_cell_source_sha256',data=sgn_line_code_sha,dtype=h5py.string_dtype())
    p.create_dataset('executed_append_cell_source',data=sgn_line_source,dtype=h5py.string_dtype())
    p.create_dataset('notebook_source_sha256',data=hashlib.sha256('\n\n'.join(c.source for c in sgn_line_notebook.cells if c.cell_type=='code').encode()).hexdigest(),dtype=h5py.string_dtype())
    write_sgn_line_value(p,'prior_outputs_sha256_unchanged',sgn_line_before)
    write_sgn_line_value(p,'output_sha256',{ext:_sgn_line_digest(SGN_LINE_FIGURE_DIR/f'SGN-lines.{ext}') for ext in ('png','pdf')})
    write_sgn_line_value(p,'runtime',{'python':sys.version,'platform':platform.platform(),'notebook_scope':'Only this new appended cell executed; prior B/C/D cells and artifacts retained'})
sgn_line_pending.replace(SGN_LINE_RESULT)
assert sorted(p.name for p in _release_cleanup_target(SGN_LINE_RESULT.parent).iterdir() if p.is_file())==['Activity.h5','Fig4B-D.h5','SGN-lines.h5']
print('SGN lines: 7 genes, 338 library values, 91 curve points; PNG/PDF PASS; prior outputs unchanged.')
# Set this storage path independently before running. SGN-lines.png.
display(Image(filename=str(ROOT),width=1400))
sgn_line_removed=0
for p in sorted(_release_cleanup_target(SGN_LINE_WORK).rglob('*'),key=lambda p:len(p.parts),reverse=True):
    assert p.resolve().is_relative_to(SGN_LINE_WORK.resolve())
    if p.is_file():sgn_line_removed+=p.stat().st_size;_release_cleanup_target(p).unlink()
    elif p.is_dir() and not any(_release_cleanup_target(p).iterdir()):_release_cleanup_target(p).rmdir()
if SGN_LINE_WORK.exists() and not any(_release_cleanup_target(SGN_LINE_WORK).iterdir()):_release_cleanup_target(SGN_LINE_WORK).rmdir()
print('Consumed runtime bytes removed:',sgn_line_removed)


## Original standalone source: fig4_validate_figure_output.py

This complete existing program retains its original command-line entry point and runtime assumptions. Use it in its original independent script context. Executing the cell in Jupyter would invoke the unchanged entry point with Jupyter arguments; this section is not a new notebook dispatcher.


In [ ]:
#!/usr/bin/env python3
"""Fail-closed validation for 600-DPI PNGs and editable figure PDFs.

The default PDF profile is intentionally strict. It is designed for figures that
must remain straightforward to edit after opening in Adobe Illustrator:

* no more than four explicit, nonzero-winding clipping paths per page;
* clipping paths are sibling, axis-aligned rectangles (never nested);
* every clipping path is explicitly scoped by q/Q and ends with W n;
* no raster images, soft/image masks, inline images, text clipping, shading,
  or uninspected recursive Form XObjects;
* live, embedded, Unicode-mapped Arial/SimSun text; and
* at least one painted vector path.

The program writes its report to stdout only. Poppler render files are created in
a task-local temporary directory and removed before exit unless the caller uses
``--render-dir``; that option always creates and retains a new unique child
directory, so existing files are never overwritten.
"""

from __future__ import annotations

import argparse
import math
import os
import re
import secrets
import shutil
import subprocess
import sys
import tempfile
from collections import Counter
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Iterable, Sequence

try:
    from PIL import Image
    from pypdf import PdfReader
    from pypdf.generic import ArrayObject, ContentStream, DictionaryObject, IndirectObject
except ImportError as exc:  # pragma: no cover - exercised only in a broken runtime
    print(f"[NOT CERTIFIED] Required Python dependency is unavailable: {exc}")
    raise SystemExit(1)


EPSILON = 1e-7
PATH_BUILDING = {b"m", b"l", b"c", b"v", b"y", b"h", b"re"}
PATH_ENDING = {b"S", b"s", b"f", b"F", b"f*", b"B", b"B*", b"b", b"b*", b"n"}
TEXT_SHOWING = {b"Tj", b"TJ", b"'", b'"'}
FILLING = {b"f", b"F", b"f*"}
STROKING = {b"S", b"s"}
FILL_STROKING = {b"B", b"B*", b"b", b"b*"}
INLINE_IMAGE_OPS = {b"BI", b"ID", b"EI", b"INLINE IMAGE"}
SUBSET_PREFIX = re.compile(r"^[A-Z]{6}\+")
CJK_RE = re.compile(
    "[\u3400-\u4dbf\u4e00-\u9fff\uf900-\ufaff\U00020000-\U0002fa1f]"
)
ASCII_ALNUM_RE = re.compile(r"[A-Za-z0-9]")


@dataclass
class Issue:
    code: str
    message: str
    outcome: str = "FAIL"


@dataclass
class ClipEvent:
    number: int
    scope: str
    depth: int
    bbox: tuple[float, float, float, float] | None
    kinds: Counter[str] = field(default_factory=Counter)


@dataclass
class FontUse:
    scope: str
    resource_name: str
    base_font: str
    subtype: str
    embedded: bool
    to_unicode: bool


@dataclass
class GraphicsState:
    ctm: tuple[float, float, float, float, float, float] = (1, 0, 0, 1, 0, 0)
    clip_depth: int = 0
    active_clips: tuple[int, ...] = ()
    font_resource: str | None = None
    font_object: Any = None
    text_render_mode: int = 0


@dataclass
class PdfPageStats:
    page_number: int
    crop_size_points: tuple[float, float] = (0.0, 0.0)
    media_size_points: tuple[float, float] = (0.0, 0.0)
    rotation: int = 0
    clip_events: list[ClipEvent] = field(default_factory=list)
    max_clip_depth: int = 0
    q_count: int = 0
    Q_count: int = 0
    form_calls: int = 0
    image_count: int = 0
    inline_image_count: int = 0
    text_show_count: int = 0
    text_object_count: int = 0
    vector_paint_count: int = 0
    vector_ops: Counter[str] = field(default_factory=Counter)
    font_uses: dict[tuple[str, str, str], FontUse] = field(default_factory=dict)


@dataclass
class ValidationResult:
    path: Path
    kind: str
    issues: list[Issue] = field(default_factory=list)
    details: list[str] = field(default_factory=list)

    @property
    def passed(self) -> bool:
        return not self.issues

    @property
    def status(self) -> str:
        if any(issue.outcome == "FAIL" for issue in self.issues):
            return "FAIL"
        if self.issues:
            return "NOT CERTIFIED"
        return "PASS"

    def fail(self, code: str, message: str) -> None:
        self.issues.append(Issue(code, message, "FAIL"))

    def not_certified(self, code: str, message: str) -> None:
        self.issues.append(Issue(code, message, "NOT CERTIFIED"))


def dereference(value: Any) -> Any:
    try:
        return value.get_object()
    except AttributeError:
        return value


def dictionary(value: Any) -> DictionaryObject | dict:
    value = dereference(value)
    return value if isinstance(value, (DictionaryObject, dict)) else {}


def numeric(value: Any) -> float:
    return float(dereference(value))


def normalize_font_name(value: Any) -> str:
    name = str(value or "").lstrip("/")
    return SUBSET_PREFIX.sub("", name)


def font_family(base_font: str) -> str | None:
    compact = re.sub(r"[^a-z0-9]", "", normalize_font_name(base_font).lower())
    if compact.startswith("arial"):
        return "Arial"
    if compact.startswith("simsun") or compact.startswith("nsimsun"):
        return "SimSun"
    return None


def font_descriptor(font_obj: Any) -> dict:
    font = dictionary(font_obj)
    subtype = str(font.get("/Subtype", ""))
    if subtype == "/Type0":
        descendants = dereference(font.get("/DescendantFonts", []))
        if descendants:
            descendant = dictionary(descendants[0])
            return dictionary(descendant.get("/FontDescriptor", {}))
    return dictionary(font.get("/FontDescriptor", {}))


def font_is_embedded(font_obj: Any) -> bool:
    descriptor = font_descriptor(font_obj)
    return any(key in descriptor for key in ("/FontFile", "/FontFile2", "/FontFile3"))


def object_identity(value: Any) -> tuple[Any, ...]:
    if isinstance(value, IndirectObject):
        return ("indirect", value.idnum, value.generation)
    obj = dereference(value)
    ref = getattr(obj, "indirect_reference", None)
    if isinstance(ref, IndirectObject):
        return ("indirect", ref.idnum, ref.generation)
    return ("direct", id(obj))


def matrix_multiply(
    first: tuple[float, float, float, float, float, float],
    second: tuple[float, float, float, float, float, float],
) -> tuple[float, float, float, float, float, float]:
    """Compose affine transforms so result(point) == first(second(point))."""
    a1, b1, c1, d1, e1, f1 = first
    a2, b2, c2, d2, e2, f2 = second
    return (
        a1 * a2 + c1 * b2,
        b1 * a2 + d1 * b2,
        a1 * c2 + c1 * d2,
        b1 * c2 + d1 * d2,
        a1 * e2 + c1 * f2 + e1,
        b1 * e2 + d1 * f2 + f1,
    )


def transform_point(
    matrix: tuple[float, float, float, float, float, float], x: float, y: float
) -> tuple[float, float]:
    a, b, c, d, e, f = matrix
    return (a * x + c * y + e, b * x + d * y + f)


def normalized_bbox(points: Sequence[tuple[float, float]]) -> tuple[float, float, float, float]:
    xs = [point[0] for point in points]
    ys = [point[1] for point in points]
    left, right = min(xs), max(xs)
    bottom, top = min(ys), max(ys)
    return (left, bottom, right - left, top - bottom)


def nearly_equal(left: float, right: float, tolerance: float) -> bool:
    return abs(left - right) <= tolerance


def boxes_match(
    observed: tuple[float, float, float, float],
    expected: tuple[float, float, float, float],
) -> bool:
    tolerance = max(0.5, max(abs(value) for value in expected[2:]) * 1e-4)
    return all(nearly_equal(a, b, tolerance) for a, b in zip(observed, expected))


def image_blank_reason(image: Image.Image) -> str | None:
    """Return a concrete blank-page reason without allocating a full RGBA copy."""
    extrema: Any = image.getextrema()
    if not isinstance(extrema, tuple):
        extrema = (extrema,)
    elif extrema and not isinstance(extrema[0], tuple):
        extrema = (extrema,)
    bands = image.getbands()
    if "A" in bands and extrema[bands.index("A")] == (0, 0):
        return "all pixels are fully transparent"
    if all(low == high for low, high in extrema):
        return "all pixels have one uniform color"
    return None


def parse_simple_rectangle(
    path_ops: Sequence[tuple[bytes, list[Any]]],
    ctm: tuple[float, float, float, float, float, float],
) -> tuple[tuple[float, float, float, float] | None, str | None]:
    if len(path_ops) == 1 and path_ops[0][0] == b"re":
        operands = path_ops[0][1]
        if len(operands) != 4:
            return None, "rectangle operator does not have four operands"
        x, y, width, height = (numeric(value) for value in operands)
        if abs(width) <= EPSILON or abs(height) <= EPSILON:
            return None, "rectangle has zero width or height"
        raw_points = [
            (x, y),
            (x + width, y),
            (x + width, y + height),
            (x, y + height),
        ]
    else:
        operators = [operator for operator, _ in path_ops]
        if not operators or operators[0] != b"m" or any(
            operator not in {b"m", b"l", b"h"} for operator in operators
        ):
            return None, "clip path is not one re rectangle or one m/l/h rectangle"
        if operators.count(b"m") != 1 or operators.count(b"h") > 1:
            return None, "clip path contains multiple subpaths"
        raw_points: list[tuple[float, float]] = []
        for operator, operands in path_ops:
            if operator in {b"m", b"l"}:
                if len(operands) != 2:
                    return None, "rectangle vertex does not have two coordinates"
                raw_points.append((numeric(operands[0]), numeric(operands[1])))
        if len(raw_points) == 5 and all(
            nearly_equal(a, b, 1e-6) for a, b in zip(raw_points[0], raw_points[-1])
        ):
            raw_points.pop()
        if len(raw_points) != 4 or (operators[-1] != b"h" and len(path_ops) != 5):
            return None, "m/l/h clip does not contain exactly four rectangle vertices"

    points = [transform_point(ctm, x, y) for x, y in raw_points]
    tolerance = max(1e-5, max(abs(value) for point in points for value in point) * 1e-7)
    for start, end in zip(points, points[1:] + points[:1]):
        dx, dy = abs(end[0] - start[0]), abs(end[1] - start[1])
        if not (dx <= tolerance < dy or dy <= tolerance < dx):
            return None, "clip rectangle is rotated, skewed, or self-degenerate"
    unique_x = []
    unique_y = []
    for x, y in points:
        if not any(nearly_equal(x, item, tolerance) for item in unique_x):
            unique_x.append(x)
        if not any(nearly_equal(y, item, tolerance) for item in unique_y):
            unique_y.append(y)
    if len(unique_x) != 2 or len(unique_y) != 2:
        return None, "clip path does not form one axis-aligned rectangle"
    return normalized_bbox(points), None


class PdfAuditor:
    def __init__(
        self,
        path: Path,
        result: ValidationResult,
        max_clips: int,
        expected_clips: int | None,
        declared_clip_boxes: list[tuple[float, float, float, float]],
        allow_raster_images: bool,
        allow_form_xobjects: bool,
        allow_no_text: bool,
        allow_no_vector: bool,
    ) -> None:
        self.path = path
        self.result = result
        self.max_clips = max_clips
        self.expected_clips = expected_clips
        self.declared_clip_boxes = declared_clip_boxes
        self.allow_raster_images = allow_raster_images
        self.allow_form_xobjects = allow_form_xobjects
        self.allow_no_text = allow_no_text
        self.allow_no_vector = allow_no_vector
        self.reader: PdfReader | None = None
        self._scanned_resources: set[tuple[Any, ...]] = set()
        self._reported_fonts: set[tuple[Any, ...]] = set()

    def audit(self) -> list[PdfPageStats]:
        try:
            self.reader = PdfReader(str(self.path), strict=True)
        except Exception as exc:
            self.result.not_certified("PDF_PARSE", f"PDF cannot be parsed strictly: {exc}")
            return []
        if self.reader.is_encrypted:
            self.result.not_certified("PDF_ENCRYPTED", "encrypted PDFs cannot be inspected")
            return []
        if not self.reader.pages:
            self.result.fail("PDF_EMPTY", "PDF contains no pages")
            return []

        page_stats: list[PdfPageStats] = []
        for page_number, page in enumerate(self.reader.pages, start=1):
            crop_box = page.cropbox
            media_box = page.mediabox
            user_unit = float(page.get("/UserUnit", 1) or 1)
            rotation = int(page.get("/Rotate", 0) or 0) % 360
            stats = PdfPageStats(
                page_number,
                crop_size_points=(
                    float(crop_box.width) * user_unit,
                    float(crop_box.height) * user_unit,
                ),
                media_size_points=(
                    float(media_box.width) * user_unit,
                    float(media_box.height) * user_unit,
                ),
                rotation=rotation,
            )
            page_stats.append(stats)
            resources = dictionary(page.get("/Resources", {}))
            page_boxes = [
                (
                    float(crop_box.left),
                    float(crop_box.bottom),
                    float(crop_box.width),
                    float(crop_box.height),
                ),
                (
                    float(media_box.left),
                    float(media_box.bottom),
                    float(media_box.width),
                    float(media_box.height),
                ),
            ]
            allowed_boxes = page_boxes + self.declared_clip_boxes
            contents = page.get_contents()
            if contents is None:
                self.result.fail("PDF_NO_CONTENT", f"page {page_number} has no content stream")
                continue
            try:
                content_stream = ContentStream(contents, self.reader)
            except Exception as exc:
                self.result.not_certified(
                    "CONTENT_PARSE", f"page {page_number} content stream cannot be parsed: {exc}"
                )
                continue
            self._scan_resources(resources, f"page {page_number}", stats)
            self._analyze_operations(
                content_stream.operations,
                resources,
                f"page {page_number}",
                stats,
                GraphicsState(),
                allowed_boxes,
                form_stack=set(),
                implicit_scope=False,
            )
            self._validate_page_text(page, stats)
            self._validate_page_summary(stats)
        return page_stats

    def _scan_resources(self, resources: dict, scope: str, stats: PdfPageStats) -> None:
        identity = object_identity(resources)
        if identity in self._scanned_resources:
            return
        self._scanned_resources.add(identity)

        ext_states = dictionary(resources.get("/ExtGState", {}))
        for name, reference in ext_states.items():
            ext_state = dictionary(reference)
            if "/SMask" in ext_state and str(dereference(ext_state["/SMask"])) != "/None":
                self.result.fail(
                    "SOFT_MASK", f"{scope} ExtGState {name} contains a non-None /SMask"
                )

        patterns = dictionary(resources.get("/Pattern", {}))
        if patterns:
            self.result.fail(
                "PATTERN_RESOURCE",
                f"{scope} contains Pattern resources; their Illustrator editability is not certified",
            )
        shadings = dictionary(resources.get("/Shading", {}))
        if shadings:
            self.result.fail(
                "SHADING_RESOURCE",
                f"{scope} contains Shading resources; their Illustrator editability is not certified",
            )

        xobjects = dictionary(resources.get("/XObject", {}))
        for name, reference in xobjects.items():
            xobject = dictionary(reference)
            subtype = str(xobject.get("/Subtype", ""))
            if "/SMask" in xobject:
                self.result.fail("SOFT_MASK", f"{scope} XObject {name} contains /SMask")
            if "/Mask" in xobject:
                self.result.fail("IMAGE_MASK", f"{scope} XObject {name} contains /Mask")
            if bool(xobject.get("/ImageMask", False)):
                self.result.fail("IMAGE_MASK", f"{scope} XObject {name} is an image mask")
            if subtype == "/Image":
                stats.image_count += 1
                if not self.allow_raster_images:
                    self.result.fail("RASTER_IMAGE", f"{scope} contains image XObject {name}")
            elif subtype == "/Form" and not self.allow_form_xobjects:
                self.result.fail(
                    "FORM_XOBJECT",
                    f"{scope} contains Form XObject {name}; use --allow-form-xobjects only for an intentionally reviewed Form workflow",
                )
            elif subtype not in {"/Form", ""}:
                self.result.fail(
                    "UNSUPPORTED_XOBJECT", f"{scope} contains unsupported XObject {name} ({subtype})"
                )

    def _record_clip_content(self, stats: PdfPageStats, state: GraphicsState, kind: str) -> None:
        for event_number in state.active_clips:
            stats.clip_events[event_number - 1].kinds[kind] += 1

    def _register_font(
        self, scope: str, resource_name: str, font_reference: Any, stats: PdfPageStats
    ) -> None:
        font = dictionary(font_reference)
        base_font = normalize_font_name(font.get("/BaseFont", ""))
        subtype = str(font.get("/Subtype", ""))
        embedded = font_is_embedded(font)
        to_unicode = "/ToUnicode" in font
        key = (scope, resource_name, base_font)
        stats.font_uses[key] = FontUse(
            scope, resource_name, base_font, subtype, embedded, to_unicode
        )
        identity = object_identity(font_reference)
        if identity in self._reported_fonts:
            return
        self._reported_fonts.add(identity)
        family = font_family(base_font)
        if family is None:
            self.result.fail(
                "FONT_FAMILY",
                f"{scope} font {resource_name} uses {base_font or '<unnamed>'}; only Arial and SimSun are allowed",
            )
        if not embedded:
            self.result.fail(
                "FONT_NOT_EMBEDDED",
                f"{scope} font {resource_name} ({base_font or '<unnamed>'}) is not embedded",
            )
        if not to_unicode:
            self.result.fail(
                "FONT_NO_TOUNICODE",
                f"{scope} font {resource_name} ({base_font or '<unnamed>'}) has no /ToUnicode map",
            )

    def _analyze_operations(
        self,
        operations: Sequence[tuple[list[Any], bytes]],
        resources: dict,
        scope: str,
        stats: PdfPageStats,
        initial_state: GraphicsState,
        allowed_boxes: list[tuple[float, float, float, float]],
        form_stack: set[tuple[Any, ...]],
        implicit_scope: bool,
    ) -> None:
        state = replace(initial_state)
        base_clip_depth = initial_state.clip_depth
        stack: list[GraphicsState] = []
        path_ops: list[tuple[bytes, list[Any]]] = []
        pending_clip = False
        in_text = False
        text_objects_at_start = stats.text_object_count

        for index, (operands, operator) in enumerate(operations):
            operands = list(operands)
            if operator == b"q":
                stats.q_count += 1
                stack.append(replace(state))
                continue
            if operator == b"Q":
                stats.Q_count += 1
                if not stack:
                    self.result.fail("Q_UNDERFLOW", f"{scope} has Q without a matching q at operation {index}")
                else:
                    state = stack.pop()
                path_ops = []
                pending_clip = False
                continue
            if operator == b"cm":
                if len(operands) != 6:
                    self.result.fail("BAD_CTM", f"{scope} has malformed cm at operation {index}")
                else:
                    matrix = tuple(numeric(value) for value in operands)
                    state.ctm = matrix_multiply(state.ctm, matrix)  # type: ignore[arg-type]
                continue

            if operator in PATH_BUILDING:
                path_ops.append((operator, operands))
                stats.vector_ops[operator.decode("ascii")] += 1
                continue
            if operator in {b"W", b"W*"}:
                if pending_clip:
                    self.result.fail(
                        "CLIP_PENDING", f"{scope} has consecutive clipping operators at operation {index}"
                    )
                if operator == b"W*":
                    self.result.fail(
                        "CLIP_EVEN_ODD",
                        f"{scope} uses W* at operation {index}; the strict profile requires nonzero-winding W",
                    )
                pending_clip = True
                continue
            if operator in PATH_ENDING:
                if pending_clip:
                    if operator != b"n":
                        self.result.fail(
                            "CLIP_PAINTED_PATH",
                            f"{scope} applies clipping with W/W* followed by {operator.decode('latin1')}; strict clipping must end with n",
                        )
                    bbox, reason = parse_simple_rectangle(path_ops, state.ctm)
                    event_number = len(stats.clip_events) + 1
                    if reason:
                        self.result.fail(
                            "CLIP_NOT_RECTANGLE",
                            f"{scope} clip {event_number} is invalid: {reason}",
                        )
                    elif bbox is not None and not any(
                        boxes_match(bbox, candidate) for candidate in allowed_boxes
                    ):
                        pretty = ", ".join(f"{value:.3f}" for value in bbox)
                        self.result.fail(
                            "CLIP_BOX_UNDECLARED",
                            f"{scope} clip {event_number} box ({pretty}) is neither the page box nor a declared --clip-box",
                        )
                    if not stack and not implicit_scope:
                        self.result.fail(
                            "CLIP_NOT_SCOPED",
                            f"{scope} clip {event_number} is not enclosed by an explicit q/Q scope",
                        )
                    state.clip_depth += 1
                    state.active_clips = state.active_clips + (event_number,)
                    stats.max_clip_depth = max(stats.max_clip_depth, state.clip_depth)
                    stats.clip_events.append(
                        ClipEvent(event_number, scope, state.clip_depth, bbox)
                    )
                    if state.clip_depth > 1:
                        self.result.fail(
                            "CLIP_NESTED",
                            f"{scope} clip {event_number} reaches effective clipping depth {state.clip_depth}",
                        )
                    pending_clip = False
                if operator in FILLING:
                    stats.vector_paint_count += 1
                    self._record_clip_content(stats, state, "vector-fill")
                elif operator in STROKING:
                    stats.vector_paint_count += 1
                    self._record_clip_content(stats, state, "vector-stroke")
                elif operator in FILL_STROKING:
                    stats.vector_paint_count += 1
                    self._record_clip_content(stats, state, "vector-fill+stroke")
                path_ops = []
                continue

            if operator == b"BT":
                if in_text:
                    self.result.fail("TEXT_NESTED", f"{scope} contains nested BT operators")
                in_text = True
                stats.text_object_count += 1
                continue
            if operator == b"ET":
                if not in_text:
                    self.result.fail("TEXT_ET_UNMATCHED", f"{scope} contains ET without BT")
                in_text = False
                continue
            if operator == b"Tr":
                try:
                    state.text_render_mode = int(numeric(operands[0]))
                except Exception:
                    self.result.fail("TEXT_RENDER_MODE", f"{scope} contains malformed Tr")
                    continue
                if state.text_render_mode in {4, 5, 6, 7}:
                    self.result.fail(
                        "TEXT_CLIPPING",
                        f"{scope} uses clipping text rendering mode {state.text_render_mode}",
                    )
                continue
            if operator == b"Tf":
                if len(operands) < 2:
                    self.result.fail("FONT_SELECT", f"{scope} contains malformed Tf")
                    continue
                resource_name = str(operands[0])
                fonts = dictionary(resources.get("/Font", {}))
                font_reference = fonts.get(resource_name)
                if font_reference is None:
                    self.result.fail(
                        "FONT_MISSING", f"{scope} selects missing font resource {resource_name}"
                    )
                    state.font_resource = None
                    state.font_object = None
                else:
                    state.font_resource = resource_name
                    state.font_object = font_reference
                continue
            if operator in TEXT_SHOWING:
                stats.text_show_count += 1
                if not in_text:
                    self.result.fail("TEXT_OUTSIDE_BT", f"{scope} shows text outside BT/ET")
                if state.text_render_mode in {3, 7}:
                    self.result.fail(
                        "INVISIBLE_TEXT", f"{scope} contains invisible text rendering mode {state.text_render_mode}"
                    )
                if state.text_render_mode in {4, 5, 6, 7}:
                    self.result.fail(
                        "TEXT_CLIPPING",
                        f"{scope} shows text in clipping rendering mode {state.text_render_mode}",
                    )
                if not state.font_resource or state.font_object is None:
                    self.result.fail("TEXT_NO_FONT", f"{scope} shows text without a resolved font")
                else:
                    self._register_font(scope, state.font_resource, state.font_object, stats)
                self._record_clip_content(stats, state, "text")
                continue

            if operator == b"Do":
                if not operands:
                    self.result.fail("XOBJECT_CALL", f"{scope} has malformed Do operator")
                    continue
                resource_name = str(operands[0])
                xobjects = dictionary(resources.get("/XObject", {}))
                reference = xobjects.get(resource_name)
                if reference is None:
                    self.result.fail(
                        "XOBJECT_MISSING", f"{scope} invokes missing XObject {resource_name}"
                    )
                    continue
                xobject = dictionary(reference)
                subtype = str(xobject.get("/Subtype", ""))
                if subtype == "/Image":
                    self._record_clip_content(stats, state, "image")
                    continue
                if subtype != "/Form":
                    self.result.fail(
                        "UNSUPPORTED_XOBJECT",
                        f"{scope} invokes unsupported XObject {resource_name} ({subtype})",
                    )
                    continue
                stats.form_calls += 1
                self._record_clip_content(stats, state, "form")
                identity = object_identity(reference)
                if identity in form_stack:
                    self.result.fail(
                        "FORM_RECURSION", f"{scope} has recursive Form XObject {resource_name}"
                    )
                    continue
                form_resources = dictionary(xobject.get("/Resources", resources))
                self._scan_resources(form_resources, f"{scope}/Form {resource_name}", stats)
                form_matrix_raw = dereference(xobject.get("/Matrix", [1, 0, 0, 1, 0, 0]))
                try:
                    form_matrix = tuple(numeric(value) for value in form_matrix_raw)
                    if len(form_matrix) != 6:
                        raise ValueError
                except Exception:
                    self.result.fail(
                        "FORM_MATRIX", f"{scope}/Form {resource_name} has malformed /Matrix"
                    )
                    form_matrix = (1, 0, 0, 1, 0, 0)
                form_state = replace(state)
                form_state.ctm = matrix_multiply(state.ctm, form_matrix)  # type: ignore[arg-type]
                try:
                    form_stream = ContentStream(dereference(reference), self.reader)
                except Exception as exc:
                    self.result.not_certified(
                        "FORM_PARSE", f"{scope}/Form {resource_name} cannot be parsed: {exc}"
                    )
                    continue
                self._analyze_operations(
                    form_stream.operations,
                    form_resources,
                    f"{scope}/Form {resource_name}",
                    stats,
                    form_state,
                    allowed_boxes,
                    form_stack | {identity},
                    implicit_scope=True,
                )
                continue

            if operator in INLINE_IMAGE_OPS:
                stats.inline_image_count += 1
                self.result.fail("INLINE_IMAGE", f"{scope} contains an inline image")
                self._record_clip_content(stats, state, "inline-image")
                continue
            if operator == b"sh":
                self.result.fail("SHADING", f"{scope} paints a shading object")
                self._record_clip_content(stats, state, "shading")
                continue

        if pending_clip:
            self.result.fail("CLIP_UNFINISHED", f"{scope} ends before its W/W* clip is applied")
        if path_ops:
            self.result.fail("PATH_UNFINISHED", f"{scope} ends with an unfinished path")
        if in_text:
            self.result.fail("TEXT_UNFINISHED", f"{scope} ends before ET")
        if stack:
            self.result.fail(
                "Q_UNBALANCED", f"{scope} has {len(stack)} q operator(s) without matching Q"
            )
        if state.clip_depth != base_clip_depth and not implicit_scope:
            self.result.fail(
                "CLIP_NOT_RESTORED",
                f"{scope} ends at clipping depth {state.clip_depth}, expected {base_clip_depth}",
            )
        if stats.text_object_count < text_objects_at_start:
            self.result.not_certified("INTERNAL_TEXT_COUNT", f"{scope} text-object accounting failed")

    def _validate_page_text(self, page: Any, stats: PdfPageStats) -> None:
        fragments: list[tuple[str, float, float, float, str]] = []
        parsed_font_names = {
            normalize_font_name(font.base_font) for font in stats.font_uses.values()
        }

        def visitor(text: str, cm: Sequence[float], tm: Sequence[float], font: Any, size: float) -> None:
            cleaned = text.strip()
            if not cleaned:
                return
            base_font = normalize_font_name(dictionary(font).get("/BaseFont", ""))
            try:
                x = float(cm[0]) * float(tm[4]) + float(cm[2]) * float(tm[5]) + float(cm[4])
                y = float(cm[1]) * float(tm[4]) + float(cm[3]) * float(tm[5]) + float(cm[5])
            except Exception:
                x = y = 0.0
            fragments.append((cleaned, x, y, float(size or 0), base_font))
            family = font_family(base_font)
            # pypdf can report a parent-page default font for text inside a Form
            # XObject. The content-stream pass above resolves the Form's actual
            # font. Apply script-specific checks only when the visitor font is a
            # font that the operation parser observed on a text-show operation.
            visitor_font_is_resolved = base_font in parsed_font_names
            if visitor_font_is_resolved and ASCII_ALNUM_RE.search(cleaned) and family != "Arial":
                self.result.fail(
                    "ASCII_FONT",
                    f"page {stats.page_number} text {cleaned!r} uses {base_font or '<unknown>'}, not Arial",
                )
            if visitor_font_is_resolved and CJK_RE.search(cleaned) and family != "SimSun":
                self.result.fail(
                    "CJK_FONT",
                    f"page {stats.page_number} Chinese text {cleaned!r} uses {base_font or '<unknown>'}, not SimSun",
                )

        try:
            page.extract_text(visitor_text=visitor)
        except Exception as exc:
            self.result.not_certified(
                "TEXT_EXTRACT", f"page {stats.page_number} live text cannot be decoded: {exc}"
            )
            return

        # Detect the common bad export in which a word or label becomes one text
        # object per character. Single tick labels are excluded by requiring a run.
        single_fragments = [
            item
            for item in fragments
            if len(item[0]) == 1 and (item[0].isalpha() or CJK_RE.search(item[0]))
        ]
        for index, current in enumerate(single_fragments):
            run = [current]
            for candidate in single_fragments[index + 1 :]:
                prior = run[-1]
                same_line = abs(candidate[2] - prior[2]) <= max(0.75, prior[3] * 0.35)
                same_font = candidate[4] == prior[4] and abs(candidate[3] - prior[3]) <= 0.25
                gap = candidate[1] - prior[1]
                nearby = -0.25 * max(prior[3], 1) <= gap <= 2.5 * max(prior[3], 1)
                if same_line and same_font and nearby:
                    run.append(candidate)
                    if len(run) >= 3:
                        word = "".join(item[0] for item in run)
                        self.result.fail(
                            "TEXT_FRAGMENTED",
                            f"page {stats.page_number} appears to split label {word!r} into individual character objects",
                        )
                        return
                elif candidate[2] < prior[2] - max(1.0, prior[3]):
                    break

    def _validate_page_summary(self, stats: PdfPageStats) -> None:
        clip_count = len(stats.clip_events)
        if clip_count > self.max_clips:
            self.result.fail(
                "TOO_MANY_CLIPS",
                f"page {stats.page_number} contains {clip_count} clipping paths; maximum is {self.max_clips}",
            )
        if self.expected_clips is not None and clip_count != self.expected_clips:
            self.result.fail(
                "CLIP_COUNT_EXPECTED",
                f"page {stats.page_number} contains {clip_count} clipping paths; expected exactly {self.expected_clips}",
            )
        if stats.max_clip_depth > 1:
            self.result.fail(
                "CLIP_DEPTH",
                f"page {stats.page_number} reaches clipping depth {stats.max_clip_depth}; maximum is 1",
            )
        for event in stats.clip_events:
            if not event.kinds:
                self.result.fail(
                    "EMPTY_CLIP",
                    f"page {stats.page_number} clip {event.number} contains no painted vector or live text content",
                )
        if stats.q_count != stats.Q_count:
            self.result.fail(
                "Q_COUNT",
                f"page {stats.page_number} recursively contains q={stats.q_count}, Q={stats.Q_count}",
            )
        if not self.allow_no_text and stats.text_show_count == 0:
            self.result.fail("NO_LIVE_TEXT", f"page {stats.page_number} contains no live text")
        if not self.allow_no_vector and stats.vector_paint_count == 0:
            self.result.fail("NO_VECTOR", f"page {stats.page_number} contains no painted vector path")


def validate_png(
    path: Path,
    target_dpi: float,
    dpi_tolerance: float,
    expected_size: tuple[float, float] | None,
) -> ValidationResult:
    result = ValidationResult(path, "PNG")
    if path.suffix.lower() != ".png":
        result.fail("PNG_EXTENSION", "filename does not end in .png")
    try:
        with Image.open(path) as image:
            if image.format != "PNG":
                result.fail("PNG_FORMAT", f"file content is {image.format}, not PNG")
            width, height = image.size
            dpi = image.info.get("dpi")
            image.load()
            blank_reason = image_blank_reason(image)
            if blank_reason:
                result.fail("PNG_BLANK", blank_reason)
    except Exception as exc:
        result.not_certified("PNG_DECODE", f"PNG cannot be decoded: {exc}")
        return result

    result.details.append(f"pixels={width}x{height}")
    if not dpi or len(dpi) < 2:
        result.fail("PNG_DPI_MISSING", "PNG has no two-axis physical-resolution metadata")
    else:
        x_dpi, y_dpi = float(dpi[0]), float(dpi[1])
        result.details.append(f"dpi={x_dpi:.3f}x{y_dpi:.3f}")
        if abs(x_dpi - target_dpi) > dpi_tolerance or abs(y_dpi - target_dpi) > dpi_tolerance:
            result.fail(
                "PNG_DPI",
                f"PNG resolution is {x_dpi:.3f}x{y_dpi:.3f} DPI; expected {target_dpi:g}+/-{dpi_tolerance:g}",
            )
        if expected_size:
            expected_width = expected_size[0] * target_dpi
            expected_height = expected_size[1] * target_dpi
            pixel_tolerance = max(1.0, target_dpi * 0.005)
            if abs(width - expected_width) > pixel_tolerance or abs(height - expected_height) > pixel_tolerance:
                result.fail(
                    "PNG_PIXEL_SIZE",
                    f"pixels {width}x{height} do not match {expected_size[0]:g}x{expected_size[1]:g} inches at {target_dpi:g} DPI",
                )
            result.details.append(
                f"expected-physical-size={expected_size[0]:g}x{expected_size[1]:g}in"
            )
        else:
            result.details.append(
                f"inferred-physical-size={width / x_dpi:.3f}x{height / y_dpi:.3f}in"
            )
    return result


def find_pdftoppm(explicit: str | None) -> str | None:
    if explicit:
        candidate = Path(explicit)
        return str(candidate) if candidate.is_file() else None
    return shutil.which("pdftoppm")


def _render_size_matches(
    observed: tuple[int, int], expected_points: tuple[float, float], dpi: float, rotation: int
) -> bool:
    width_points, height_points = expected_points
    if rotation in {90, 270}:
        width_points, height_points = height_points, width_points
    expected = (width_points * dpi / 72.0, height_points * dpi / 72.0)
    # Poppler normally rounds page boundaries outward. Accept at most one pixel
    # of normal raster-grid rounding in either dimension.
    return all(abs(actual - target) <= 1.05 for actual, target in zip(observed, expected))


def _prepare_render_directory(
    path: Path, result: ValidationResult, requested_base: Path | None
) -> tuple[Path | None, tempfile.TemporaryDirectory[str] | None]:
    if requested_base is not None:
        try:
            base = requested_base.expanduser().resolve(strict=False)
            if base.exists() and not base.is_dir():
                result.not_certified(
                    "RENDER_DIR_NOT_DIRECTORY", f"--render-dir is not a directory: {base}"
                )
                return None, None
            base.mkdir(parents=True, exist_ok=True)
            safe_stem = re.sub(r"[^A-Za-z0-9._-]+", "-", path.stem).strip("-.") or "figure"
            # Always create a normal child directory so Windows applies the
            # parent's inherited ACL. Path.mkdir(exist_ok=False) is atomic and
            # cannot overwrite an existing entry; the retry loop handles the
            # extremely unlikely random-name collision.
            output = None
            for _ in range(20):
                candidate = base / f"{safe_stem[:60]}-render-{secrets.token_hex(4)}"
                try:
                    candidate.mkdir(exist_ok=False)
                except FileExistsError:
                    continue
                output = candidate
                break
            if output is None:
                result.not_certified(
                    "RENDER_DIR_COLLISION",
                    "could not allocate a unique render child after 20 attempts",
                )
                return None, None
        except OSError as exc:
            result.not_certified(
                "RENDER_DIR_CREATE", f"cannot create a unique child under --render-dir: {exc}"
            )
            return None, None
        result.details.append(f"render-output-dir={output} (retained; unique child, no overwrite)")
        return output, None

    task_tmp_root = Path.cwd() / "tmp"
    try:
        task_tmp_root.mkdir(parents=True, exist_ok=True)
        temporary = tempfile.TemporaryDirectory(prefix="validate-figure-", dir=task_tmp_root)
    except OSError:
        try:
            temporary = tempfile.TemporaryDirectory(prefix="validate-figure-")
        except OSError as exc:
            result.not_certified("RENDER_DIR_CREATE", f"cannot create a temporary render directory: {exc}")
            return None, None
    return Path(temporary.name), temporary


def render_pdf(
    path: Path,
    result: ValidationResult,
    page_stats: Sequence[PdfPageStats],
    executable: str | None,
    render_dpi: float,
    render_dir: Path | None,
) -> None:
    pdftoppm = find_pdftoppm(executable)
    if not pdftoppm:
        result.not_certified("POPPLER_MISSING", "pdftoppm was not found; visual rendering is mandatory")
        return
    output_dir, temporary = _prepare_render_directory(path, result, render_dir)
    if output_dir is None:
        return
    try:
        prefix = output_dir / "page"
        command = [
            pdftoppm,
            "-png",
            "-r",
            f"{render_dpi:g}",
            "-f",
            "1",
            "-l",
            str(len(page_stats)),
            str(path),
            str(prefix),
        ]
        try:
            completed = subprocess.run(
                command,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                timeout=max(60, len(page_stats) * 30),
                check=False,
            )
        except Exception as exc:
            result.not_certified("POPPLER_RUN", f"Poppler rendering could not run: {exc}")
            return
        if completed.returncode != 0:
            detail = completed.stderr.strip() or f"exit code {completed.returncode}"
            result.not_certified("POPPLER_RENDER", f"Poppler failed to render the PDF: {detail}")
            return
        rendered: dict[int, Path] = {}
        for candidate in output_dir.glob("page-*.png"):
            match = re.fullmatch(r"page-(\d+)\.png", candidate.name, flags=re.IGNORECASE)
            if match:
                rendered[int(match.group(1))] = candidate
        if len(rendered) != len(page_stats) or set(rendered) != set(
            range(1, len(page_stats) + 1)
        ):
            result.not_certified(
                "POPPLER_PAGE_COUNT",
                f"Poppler produced page numbers {sorted(rendered)}, expected 1..{len(page_stats)}",
            )
            return
        for stats in page_stats:
            output = rendered[stats.page_number]
            try:
                with Image.open(output) as image:
                    image.verify()
                with Image.open(output) as image:
                    image.load()
                    observed_size = image.size
                    blank_reason = image_blank_reason(image)
            except Exception as exc:
                result.not_certified("POPPLER_OUTPUT", f"rendered page {output.name} is invalid: {exc}")
                continue
            if stats.rotation not in {0, 90, 180, 270}:
                result.not_certified(
                    "PAGE_ROTATION",
                    f"page {stats.page_number} has unsupported /Rotate={stats.rotation}",
                )
                matched_box = None
            elif _render_size_matches(
                observed_size, stats.crop_size_points, render_dpi, stats.rotation
            ):
                matched_box = "CropBox"
            elif _render_size_matches(
                observed_size, stats.media_size_points, render_dpi, stats.rotation
            ):
                matched_box = "MediaBox"
            else:
                matched_box = None
                result.fail(
                    "RENDER_PIXEL_SIZE",
                    f"page {stats.page_number} rendered to {observed_size[0]}x{observed_size[1]} px at {render_dpi:g} DPI; "
                    f"this matches neither CropBox {stats.crop_size_points[0]:.3f}x{stats.crop_size_points[1]:.3f} pt "
                    f"nor MediaBox {stats.media_size_points[0]:.3f}x{stats.media_size_points[1]:.3f} pt",
                )
            if blank_reason:
                result.fail(
                    "RENDER_BLANK",
                    f"page {stats.page_number} rendered blank: {blank_reason}",
                )
            match_text = matched_box or "no-box-match"
            content_text = "blank" if blank_reason else "nonblank"
            result.details.append(
                f"render-page={stats.page_number} pixels={observed_size[0]}x{observed_size[1]} "
                f"dpi={render_dpi:g} box={match_text} content={content_text}"
            )
        result.details.append(f"poppler-rendered-pages={len(rendered)}")
    finally:
        if temporary is not None:
            temporary.cleanup()


def validate_pdf(path: Path, args: argparse.Namespace) -> ValidationResult:
    result = ValidationResult(path, "PDF")
    if path.suffix.lower() != ".pdf":
        result.fail("PDF_EXTENSION", "filename does not end in .pdf")
    auditor = PdfAuditor(
        path,
        result,
        max_clips=args.max_clips,
        expected_clips=args.expected_clips,
        declared_clip_boxes=args.clip_box,
        allow_raster_images=args.allow_raster_images,
        allow_form_xobjects=args.allow_form_xobjects,
        allow_no_text=args.allow_no_text,
        allow_no_vector=args.allow_no_vector,
    )
    page_stats = auditor.audit()
    if page_stats:
        for stats in page_stats:
            fonts = sorted({font.base_font for font in stats.font_uses.values()})
            clip_summary = "; ".join(
                f"#{event.number} depth={event.depth} kinds={'+'.join(sorted(event.kinds)) or 'empty'}"
                for event in stats.clip_events
            )
            result.details.append(
                f"page={stats.page_number} clips={len(stats.clip_events)} max-depth={stats.max_clip_depth} "
                f"q/Q={stats.q_count}/{stats.Q_count} text-shows={stats.text_show_count} "
                f"vector-paints={stats.vector_paint_count} images={stats.image_count + stats.inline_image_count} "
                f"forms={stats.form_calls} fonts={','.join(fonts) or 'none'}"
            )
            if clip_summary:
                result.details.append(f"page={stats.page_number} clip-groups: {clip_summary}")
        render_pdf(
            path,
            result,
            page_stats,
            args.pdftoppm,
            args.render_dpi,
            args.render_dir,
        )
    return result


def parse_box(value: str) -> tuple[float, float, float, float]:
    try:
        fields = tuple(float(part.strip()) for part in value.split(","))
    except ValueError as exc:
        raise argparse.ArgumentTypeError("clip box must be x,y,width,height") from exc
    if len(fields) != 4 or fields[2] <= 0 or fields[3] <= 0:
        raise argparse.ArgumentTypeError("clip box must be x,y,width,height with positive size")
    return fields  # type: ignore[return-value]


def parse_size(value: str) -> tuple[float, float]:
    match = re.fullmatch(r"\s*([0-9]*\.?[0-9]+)\s*[xX×]\s*([0-9]*\.?[0-9]+)\s*", value)
    if not match:
        raise argparse.ArgumentTypeError("expected size must be WIDTHxHEIGHT in inches")
    return (float(match.group(1)), float(match.group(2)))


def positive_float(value: str) -> float:
    try:
        parsed = float(value)
    except ValueError as exc:
        raise argparse.ArgumentTypeError("value must be a positive number") from exc
    if not math.isfinite(parsed) or parsed <= 0:
        raise argparse.ArgumentTypeError("value must be a finite positive number")
    return parsed


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description="Validate 600-DPI PNGs or Illustrator-editable vector figure PDFs."
    )
    parser.add_argument("files", nargs="+", type=Path)
    parser.add_argument("--target-dpi", type=float, default=600.0)
    parser.add_argument("--dpi-tolerance", type=float, default=1.0)
    parser.add_argument(
        "--expected-size-in",
        type=parse_size,
        metavar="WIDTHxHEIGHT",
        help="also verify PNG pixel dimensions against a physical size in inches",
    )
    parser.add_argument("--max-clips", type=int, default=4, choices=range(0, 5))
    parser.add_argument(
        "--expected-clips",
        type=int,
        choices=range(0, 5),
        help="require exactly this many clipping paths on every PDF page (use 3 for the full-period regression profile)",
    )
    parser.add_argument(
        "--clip-box",
        type=parse_box,
        action="append",
        default=[],
        metavar="X,Y,W,H",
        help="declare an allowed non-page rectangular PDF clipping box (repeatable)",
    )
    parser.add_argument(
        "--allow-raster-images",
        action="store_true",
        help="allow intentional image XObjects; masks and inline images remain forbidden",
    )
    parser.add_argument(
        "--allow-form-xobjects",
        action="store_true",
        help="allow recursively inspected Form XObjects; the strict default rejects all Forms",
    )
    parser.add_argument("--allow-no-text", action="store_true")
    parser.add_argument("--allow-no-vector", action="store_true")
    parser.add_argument("--pdftoppm", help="explicit path to Poppler pdftoppm")
    parser.add_argument(
        "--render-dpi",
        type=positive_float,
        default=72.0,
        help="Poppler verification resolution (default: 72 DPI)",
    )
    parser.add_argument(
        "--render-dir",
        type=Path,
        help="retain rendered pages in a newly created unique child of DIR; existing contents are never overwritten",
    )
    return parser


def print_result(result: ValidationResult) -> None:
    print(f"[{result.status}] {result.path}")
    for detail in result.details:
        print(f"  INFO {detail}")
    for issue in result.issues:
        label = "ERROR" if issue.outcome == "FAIL" else "BLOCKER"
        print(f"  {label} [{issue.code}] {issue.message}")


def main(argv: Sequence[str] | None = None) -> int:
    parser = build_parser()
    args = parser.parse_args(argv)
    if args.expected_clips is not None and args.expected_clips > args.max_clips:
        parser.error("--expected-clips cannot exceed --max-clips")
    results: list[ValidationResult] = []
    for raw_path in args.files:
        path = raw_path.resolve()
        if not path.is_file():
            result = ValidationResult(path, "UNKNOWN")
            result.not_certified("FILE_MISSING", "file does not exist or is not a regular file")
        elif path.suffix.lower() == ".png":
            result = validate_png(path, args.target_dpi, args.dpi_tolerance, args.expected_size_in)
        elif path.suffix.lower() == ".pdf":
            result = validate_pdf(path, args)
        else:
            result = ValidationResult(path, "UNKNOWN")
            result.not_certified("FILE_TYPE", "only .png and .pdf are supported")
        results.append(result)
        print_result(result)
    statuses = Counter(result.status for result in results)
    passed = statuses["PASS"]
    print(
        f"SUMMARY passed={passed} failed={statuses['FAIL']} "
        f"not-certified={statuses['NOT CERTIFIED']} total={len(results)}"
    )
    return 0 if passed == len(results) else 1


if __name__ == "__main__":
    raise SystemExit(main())


## Existing panel drawing: unchanged source and layout

The following existing notebook cells are copied unchanged, including original shared figure outputs. Use this section in its original context; similarly named globals in other sections are not a new shared execution API.


# Supplementary Figure 4A: SGN-related gene expression by age

Set the paths below. This notebook reads `SGN-lines.h5`, produced by the seven-gene
age-curve step in `05_fig4B-D_SGN.ipynb` (source cell index 12).
The library points, source-balanced means, gene order, colors and plotting code
are retained from that step.


In [ ]:
#1. =========== Paths and fixed display settings
from pathlib import Path
import hashlib
import json
import math
import os
import re
import sys
import h5py
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager
from PIL import Image as PILImage

ROOT = Path('.')  # Set the working root before replacing the path placeholders.
# Uses SGN-lines.h5 from the selected-gene age-curve step in 05_fig4B-D_SGN.ipynb.
SGN_LINE_RESULT = ROOT  # Set the saved seven-gene result H5 file path.
PNG_FILE = ROOT  # Set this panel's output PNG file path.
PDF_FILE = ROOT  # Set this panel's output PDF file path.
OUTPUT_DIR = ROOT  # Set the output directory path.
WORK_DIR = ROOT  # Set the temporary working directory path.
FONT_FILES = {
    'normal': ROOT,  # Set the Arial regular TTF file path.
    'italic': ROOT,  # Set the Arial italic TTF file path.
    'bold': ROOT,  # Set the Arial bold TTF file path.
    'bolditalic': ROOT,  # Set the Arial bold italic TTF file path.
}
os.environ['MPLCONFIGDIR'] = str(WORK_DIR)
sys.dont_write_bytecode = True
for font_file in FONT_FILES.values():
    font_manager.fontManager.addfont(str(font_file))



In [ ]:
#2. =========== Read the saved seven-gene curves
from urllib.parse import unquote
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator
def read_sgn_line_value(node):
    kind = node.attrs['kind']
    if kind == 'dataframe':
        columns = json.loads(node.attrs['columns'])
        out = pd.DataFrame({key: read_sgn_line_value(node[key]) for key in columns})
        out.index = read_sgn_line_value(node['_index']); return out
    if kind == 'dict': return {unquote(key): read_sgn_line_value(node[key]) for key in node}
    if kind == 'json': return json.loads(node.asstr()[()])
    return node.asstr()[()] if node.dtype.kind in 'OUS' else node[()]

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

SGN_LINE_GENES = ('Foxo6','Zfpm2','Neurod2','Nhlh1','Neurod1','Prox1','Insm1')
SGN_LINE_FIGSIZE, SGN_LINE_DPI = (15.2,6.4),600
SGN_LINE_DATA_SOURCE_SHA256 = 'b712bd1fc4c64b302907823fc0df88428256e84c447e4a1d2a56a18f8dd2f416'
assert file_sha256(SGN_LINE_RESULT) == '698c19bfd2ded659cb52ba95e8f0c8673dc1f9b967e65599f066affea31f2a86', 'The accepted age-curve archive changed.'
with h5py.File(SGN_LINE_RESULT, 'r') as handle:
    assert handle.attrs.get('schema') == 'SGN-seven-gene-lines-v1'
    assert handle['provenance/data_source_sha256'].asstr()[()] == SGN_LINE_DATA_SOURCE_SHA256
    sgn_line_data = read_sgn_line_value(handle['data'])
np.testing.assert_array_equal(sgn_line_data['genes'], SGN_LINE_GENES)
assert len(sgn_line_data['line_units']) == 338
assert len(sgn_line_data['line_points']) == 91


In [ ]:
#3. =========== Original age-curve renderer
"""Strict vector PDF export for the Fig4 B/C/D Matplotlib canvas.

No analysis is performed. Each native Matplotlib label becomes one live Arial
text object, including the TF italic run in the two supported mathtext titles.
All geometry is written directly, without XObjects, glyph outlines or images.
The caller declares at most four data axes; their clip rectangles are collected
into one sibling viewport group per axis. Other artists must fit their axes.
"""

from pathlib import Path

import re

import sys

import math

import numpy as np

from matplotlib.backend_bases import RendererBase

from matplotlib.path import Path as MplPath

from reportlab.pdfgen import canvas

from reportlab.pdfbase import pdfmetrics

from reportlab.pdfbase.ttfonts import TTFont

_SGN_LINE_FONTS = {'normal': 'FigArial', 'italic': 'FigArialItalic', 'bold': 'FigArialBold', 'bolditalic': 'FigArialBoldItalic'}

def _sgn_line_register_fonts():
    for key, file in FONT_FILES.items():
        if _SGN_LINE_FONTS[key] not in pdfmetrics.getRegisteredFontNames():
            pdfmetrics.registerFont(TTFont(_SGN_LINE_FONTS[key], str(file)))

def _sgn_line_font(prop):
    bold = prop.get_weight() in ('bold', 'heavy', 'black', 'semibold', 'demibold') or (isinstance(prop.get_weight(), (int, float)) and prop.get_weight() >= 600)
    italic = prop.get_style() in ('italic', 'oblique')
    return _SGN_LINE_FONTS['bolditalic' if bold and italic else 'bold' if bold else 'italic' if italic else 'normal']

def _sgn_line_runs(text, prop, ismath):
    if not ismath:
        if '$' in text:
            raise ValueError('Unexpected dollar in plain label: ' + text)
        return [(text, _sgn_line_font(prop))]
    found = re.fullmatch('\\$\\\\it\\{([^{}]+)\\}([^$]*)\\$(.*)', text)
    if found is None:
        raise ValueError('Unsupported math label: ' + text)
    tf, inside, after = found.groups()
    return [(tf, _SGN_LINE_FONTS['italic']), (inside + after, _SGN_LINE_FONTS['normal'])]

class _SGNLineRecorder(RendererBase):

    def __init__(self, fig, agg, viewports, halo_labels):
        super().__init__()
        self.dpi = fig.dpi
        self.unit = 72 / fig.dpi
        self.agg = agg
        self.width, self.height = fig.bbox.size
        self.ops = []
        self.viewports = viewports
        self.halo_labels = halo_labels

    def points_to_pixels(self, p):
        return p / self.unit

    def flipy(self):
        return False

    def get_canvas_width_height(self):
        return (self.width, self.height)

    def get_text_width_height_descent(self, s, prop, ismath):
        runs = _sgn_line_runs(s, prop, ismath)
        width = sum((pdfmetrics.stringWidth(t, f, prop.get_size_in_points()) for t, f in runs)) / self.unit
        _, h, d = self.agg.get_text_width_height_descent(re.sub('\\$|\\\\it\\{|\\}', '', s), prop, False)
        return (width, h, d)

    def _clip(self, gc):
        if gc.get_clip_path()[0] is not None:
            raise ValueError('Nonrectangular artist clip is unsupported')
        b = gc.get_clip_rectangle()
        if b is None:
            return None
        for i, box in enumerate(self.viewports):
            if np.allclose(b.bounds, box, rtol=0, atol=1e-05):
                return i
        return None

    def draw_path(self, gc, path, transform, rgbFace=None):
        if gc.get_hatch():
            raise ValueError('Hatching/Pattern is forbidden')
        p = transform.transform_path(path)
        self.ops.append(('path', self._clip(gc), p, gc.get_rgb(), rgbFace, gc.get_linewidth(), gc.get_alpha(), gc.get_forced_alpha(), gc.get_dashes(), gc.get_capstyle(), gc.get_joinstyle()))

    def draw_text(self, gc, x, y, s, prop, angle, ismath=False, mtext=None):
        self.ops.append(('text', None, float(x), float(y), str(s), prop.copy(), float(angle), ismath, gc.get_rgb(), gc.get_alpha(), id(mtext) in self.halo_labels))

    def draw_image(self, *args, **kwargs):
        raise ValueError('Raster image is forbidden')

    def start_rasterizing(self):
        pass

    def stop_rasterizing(self):
        pass

    def start_filter(self):
        raise ValueError('Raster filter is forbidden')

    def stop_filter(self, *args):
        raise ValueError('Raster filter is forbidden')

def _sgn_line_pdf_path(pdf, path, unit):
    out = pdf.beginPath()
    current = None
    start = None
    for vertices, code in path.iter_segments(remove_nans=True, simplify=False, curves=True):
        v = np.asarray(vertices).reshape(-1, 2) * unit
        if code == MplPath.MOVETO:
            current = v[0]
            start = current
            out.moveTo(*current)
        elif code == MplPath.LINETO:
            current = v[0]
            out.lineTo(*current)
        elif code == MplPath.CURVE3:
            control, end = v
            a = current + (control - current) * 2 / 3
            b = end + (control - end) * 2 / 3
            out.curveTo(*a, *b, *end)
            current = end
        elif code == MplPath.CURVE4:
            out.curveTo(*v.ravel())
            current = v[-1]
        elif code == MplPath.CLOSEPOLY:
            out.close()
            current = start
        elif code == MplPath.STOP:
            break
        else:
            raise ValueError('Unknown path opcode ' + str(code))
    return out

def sgn_line_editable_pdf(fig, destination, *, data_axes=(), live_halo_width=0.2):
    """Write a strict vector PDF. Return declared clip boxes in PDF points.

    The artist canvas remains unchanged after return. Rasterization flags and
    white text path effects are replaced by a narrow live text fill+stroke
    (render mode 2) in the same single label object; no outlined font paths are
    emitted. The width is narrower than Agg's halo to preserve the black stems.
    """
    _sgn_line_register_fonts()
    if len(data_axes) > 4:
        raise ValueError('At most four sibling data viewports')
    fig.canvas.draw()
    agg = fig.canvas.get_renderer()
    viewports = [tuple(ax.bbox.bounds) for ax in data_axes]
    effects = []
    for artist in fig.findobj():
        if hasattr(artist, 'get_path_effects'):
            old = artist.get_path_effects()
            if old:
                effects.append((artist, old))
                artist.set_path_effects([])
    rec = _SGNLineRecorder(fig, agg, viewports, {id(a) for a, e in effects})
    try:
        fig.draw(rec)
    finally:
        for a, e in effects:
            a.set_path_effects(e)
    dst = Path(destination)
    dst.parent.mkdir(parents=True, exist_ok=True)
    pdf = canvas.Canvas(str(dst), pagesize=tuple(fig.get_size_inches() * 72), pageCompression=1, invariant=1, initialFontName=_SGN_LINE_FONTS['normal'])
    unit = rec.unit

    def emit(op):
        if op[0] == 'text':
            _, _, x, y, s, prop, angle, ismath, color, alpha, halo = op
            if not s:
                return
            pdf.saveState()
            pdf.setFillColorRGB(*color[:3])
            pdf.setFillAlpha(alpha)
            text = pdf.beginText()
            rad = math.radians(angle)
            if halo and live_halo_width > 0:
                pdf.setStrokeColorRGB(1, 1, 1)
                pdf.setStrokeAlpha(alpha)
                pdf.setLineWidth(float(live_halo_width))
                text.setTextRenderMode(2)
            text.setTextTransform(math.cos(rad), math.sin(rad), -math.sin(rad), math.cos(rad), x * unit, y * unit)
            for value, font in _sgn_line_runs(s, prop, ismath):
                if value:
                    text.setFont(font, prop.get_size_in_points())
                    text.textOut(value)
            pdf.drawText(text)
            pdf.restoreState()
            return
        _, _, path, stroke, face, lw, alpha, forced, dashes, cap, join = op
        pdf.saveState()
        pdf.setLineWidth(lw)
        pdf.setLineCap({'butt': 0, 'round': 1, 'projecting': 2}[cap])
        pdf.setLineJoin({'miter': 0, 'round': 1, 'bevel': 2}[join])
        offset, pattern = dashes
        pdf.setDash([] if pattern is None else [float(v) for v in pattern], 0.0 if offset is None else float(offset))
        if stroke is not None:
            pdf.setStrokeColorRGB(*stroke[:3])
            pdf.setStrokeAlpha(alpha if forced else stroke[3] if len(stroke) > 3 else alpha)
        if face is not None:
            pdf.setFillColorRGB(*face[:3])
            pdf.setFillAlpha(alpha if forced else face[3] if len(face) > 3 else alpha)
        pdf.drawPath(_sgn_line_pdf_path(pdf, path, unit), stroke=int(stroke is not None and lw > 0), fill=int(face is not None), fillMode=1)
        pdf.restoreState()
    done = set()
    for op in rec.ops:
        key = op[1]
        if key is None:
            emit(op)
            continue
        if key in done:
            continue
        done.add(key)
        pdf.saveState()
        p = pdf.beginPath()
        p.rect(*np.asarray(viewports[key]) * unit)
        pdf.clipPath(p, stroke=0, fill=0, fillMode=1)
        for candidate in rec.ops:
            if candidate[1] == key:
                emit(candidate)
        pdf.restoreState()
    pdf.showPage()
    pdf.save()
    if done != set(range(len(data_axes))):
        raise AssertionError('Declared clip has no artist')
    return {'expected_clips': len(done), 'clip_boxes_pt': (np.asarray(viewports) * unit).tolist(), 'editable_label_objects': sum((op[0] == 'text' for op in rec.ops)), 'label_strings': [op[4] for op in rec.ops if op[0] == 'text'], 'vector_paths': sum((op[0] == 'path' for op in rec.ops)), 'text_effects': 'White live text fill+stroke, single object; stroke width ' + str(live_halo_width) + ' pt.', 'halo_labels': len(effects)}

from PIL import Image as PILImage

def save_sgn_line_figure(fig, axes):
    fig.canvas.draw();renderer=fig.canvas.get_renderer()
    clipped=[]
    for text in fig.findobj(matplotlib.text.Text):
        if text.get_visible() and text.get_text():
            box=text.get_window_extent(renderer)
            if not fig.bbox.contains(*box.p0) or not fig.bbox.contains(*box.p1):clipped.append(text.get_text())
    assert not clipped,clipped
    # No data clip is needed: both centers and marker radii fit the declared axes.
    for ax in axes:
        for artist in ax.collections:
            xy=ax.transData.transform(np.asarray(artist.get_offsets(),float))
            radius=np.sqrt(np.max(artist.get_sizes()))/2*fig.dpi/72
            assert (xy[:,0]-radius >= ax.bbox.x0).all() and (xy[:,0]+radius <= ax.bbox.x1).all()
            assert (xy[:,1]-radius >= ax.bbox.y0).all() and (xy[:,1]+radius <= ax.bbox.y1).all()
        for artist in ax.lines:
            xy=ax.transData.transform(np.column_stack((artist.get_xdata(),artist.get_ydata())))
            radius=artist.get_markersize()/2*fig.dpi/72
            assert (xy[:,0]-radius >= ax.bbox.x0).all() and (xy[:,0]+radius <= ax.bbox.x1).all()
            assert (xy[:,1]-radius >= ax.bbox.y0).all() and (xy[:,1]+radius <= ax.bbox.y1).all()
    png=PNG_FILE;pdf=PDF_FILE
    fig.savefig(png,dpi=SGN_LINE_DPI,facecolor='white')
    qa=sgn_line_editable_pdf(fig,pdf,data_axes=())
    with PILImage.open(png) as im:
        assert im.size == (9120,3840)
        assert all(abs(x-SGN_LINE_DPI)<.1 for x in im.info['dpi'])
        qa.update(pixels=list(im.size),dpi=list(im.info['dpi']))
    qa.update(clipped_labels=clipped,legend_labels=['CVG','VGN','SGN'],
        colored_library_points_retained=True,figsize_in=list(SGN_LINE_FIGSIZE),
        library_alpha=.25,library_area=10.,
        palette={'CVG':'#8D77A1','VGN':'#5B85A7','SGN':'#69A18B'})
    plt.close(fig)
    return qa

def render_sgn_lines(data):
    FIGSIZE = SGN_LINE_FIGSIZE
    X_LIMITS, Y_LIMITS = (9.1, 18.9), (-.10, 4.10)
    X_TICKS, Y_TICKS = (9.5, 11.5, 13.5, 16.5, 18.5), (0, 1, 2, 3, 4)
    MEAN_LINE_WIDTH, MEAN_MARKER_SIZE = 1.65, 3.8
    LIBRARY_AREA, LIBRARY_ALPHA = 10.0, .25
    ORDER = ('CVG', 'VGN', 'SGN')
    COLORS = {'CVG': '#8D77A1', 'VGN': '#5B85A7', 'SGN': '#69A18B'}
    units, points = data['line_units'], data['line_points']
    plt.rcParams.update({'font.family': 'Arial', 'font.size': 9,
        'text.color': '#30343A', 'axes.labelcolor': '#30343A',
        'xtick.color': '#50555B', 'ytick.color': '#50555B', 'savefig.facecolor': 'white'})
    fig = plt.figure(figsize=FIGSIZE)
    grid = fig.add_gridspec(2, 4, left=.058, right=.985, bottom=.13, top=.94,
                           hspace=.53, wspace=.25)
    axes = []
    for j, gene in enumerate(SGN_LINE_GENES):
        row, col = (0, j) if j < 3 else (1, j - 3)
        ax = fig.add_subplot(grid[row, col])
        axes.append(ax)
        for identity in ORDER:
            u = units.loc[units.gene.eq(gene) & units.display_type.eq(identity)]
            p = points.loc[points.gene.eq(gene) & points.display_type.eq(identity)].sort_values('age')
            ax.scatter(u.age, u.expr, s=LIBRARY_AREA, color=COLORS[identity],
                       alpha=LIBRARY_ALPHA, linewidths=0, zorder=2)
            line, = ax.plot(p.age, p.expr, color=COLORS[identity], lw=MEAN_LINE_WIDTH,
                           marker='o', ms=MEAN_MARKER_SIZE, markeredgewidth=0, zorder=3)
            np.testing.assert_array_equal(line.get_ydata(), p.expr)
        ax.set_xlim(X_LIMITS); ax.set_ylim(Y_LIMITS)
        ax.set_xticks(X_TICKS, [f'E{x:g}' for x in X_TICKS])
        ax.xaxis.set_minor_locator(FixedLocator(np.arange(9.5, 19)))
        ax.set_yticks(Y_TICKS)
        if col: ax.tick_params(labelleft=False)
        ax.tick_params(which='major', length=2.6, width=.6, labelsize=8.5, pad=3)
        ax.tick_params(axis='x', which='minor', length=1.5, width=.45, color='#A5AAB0')
        ax.grid(axis='y', color='#EBECEE', linewidth=.55, zorder=0)
        for side in ('top', 'right'): ax.spines[side].set_visible(False)
        for side in ('left', 'bottom'):
            ax.spines[side].set_color('#A3A7AD'); ax.spines[side].set_linewidth(.65)
        ax.set_title(gene, fontstyle='italic', fontsize=12, pad=8)
    legend_ax = fig.add_subplot(grid[0, 3]); legend_ax.axis('off')
    handles = [Line2D([0], [0], color=COLORS[t], lw=MEAN_LINE_WIDTH, marker='o',
                      ms=4, markeredgewidth=0, label=t) for t in ORDER]
    line_legend = legend_ax.legend(handles=handles, loc='center', frameon=False, fontsize=9.5,
                     handlelength=2.2, labelspacing=1.0)
    fig.supylabel('RNA: mean log10(1 + pseudobulk CPM)', x=.012, y=.55, fontsize=10)
    fig.supxlabel('Embryonic age', x=.53, y=.035, fontsize=10)
    fig.canvas.draw(); renderer = fig.canvas.get_renderer()
    for ax in axes:
        boxes = [t.get_window_extent(renderer) for t in ax.get_xticklabels() if t.get_visible()]
        assert not any(a.overlaps(b) for i, a in enumerate(boxes) for b in boxes[i + 1:])
    assert [t.get_text() for t in line_legend.get_texts()] == list(ORDER)
    return save_sgn_line_figure(fig,axes)

In [ ]:
#4. =========== Export Supplementary Figure 4A
with plt.rc_context():
    figure_checks = render_sgn_lines(sgn_line_data)
